# Shear Modulus (μ) Visualization for CG2 Data

This notebook visualizes shear modulus structures saved with CG2 (quadratic) elements, with publication-quality plotting matching the original `plt_slip_mu_relation_CK.ipynb` notebook.

**Key difference from CG1:**
- CG2 stores values at vertices AND edge midpoints (~6x more sample points)
- Data is loaded as a point cloud to preserve all DOF locations
- Slicing uses tolerance-based point extraction + griddata interpolation

**Features (matching original notebook):**
- Geographic coordinates (lon/lat) with proper coordinate transformation
- Plate interface depth contours
- Trench line overlay
- Publication-quality PyGMT multi-panel figures
- Consistent colormaps (`gist_rainbow_r` for μ, `cmc.roma` for anomaly)

**Contents:**
1. Configuration with coordinate transformation parameters
2. Load μ from XDMF files (CG1 or CG2)
3. Load plate interface and trench
4. Slice extraction functions (unified for CG1/CG2)
5. Horizontal slices at different depths (matplotlib)
6. Vertical slices along dip and strike (matplotlib)
7. PyGMT publication-quality plots
8. CG1 vs CG2 comparison

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyvista as pv
import pygmt
import meshio
from scipy.interpolate import griddata, LinearNDInterpolator
import cmcrameri.cm as cmc

# Add path for local utilities
import sys
sys.path.append('/home/staff/chao/SSEinv/Nicoya/codes/')
import utils_plot as utp

# FEniCS for mesh loading (optional, only needed for uneven mesh top boundary)
try:
    import dolfin as dl
    DOLFIN_AVAILABLE = True
except ImportError:
    DOLFIN_AVAILABLE = False
    print("Warning: dolfin not available. Top boundary extraction will be skipped.")

# Disable PyVista interactive plotting for notebook
pv.set_jupyter_backend('static')

print("Imports complete")
if DOLFIN_AVAILABLE:
    print("  dolfin available for mesh loading")

## 1. Configuration

In [ ]:
# Paths
datadir = "/home/staff/chao/SSEinv/Nicoya/data/"
meshpath = "/home/staff/chao/SSEinv/Nicoya/mesh/"
# resultpath = "/home/staff/chao/SSEinv/Nicoya/syn_slip/"
resultpath = "/home/staff/chao/SSEinv/Nicoya/rst_locking/"

# Mesh name
# meshname = 'nicoyaCKden_sm'
# meshname = 'nicoyaCKden_une_sm'  # Uneven top boundary mesh
meshname = 'nicoyaCKden_une_all'  # Uneven top boundary mesh

# Auto-detect uneven mesh from name
use_uneven_mesh = "une" in meshname

# Model identifiers
# het3d_str = '_DeShon3D_ref_4'  # Heterogeneous 3D model
# het3d_str = '_DeShon3D_ref_4_hull'  # Heterogeneous 3D model
het3d_str = '_DeShon3D_ref_1_hull'  # Heterogeneous 3D model    # original, unscaled model
# het3d_str = '_DeShon3D_ref_4_hull_test1'  # Heterogeneous 3D model

homo_str = '_mul40u40'          # Homogeneous reference model

# CG degree used when saving the μ function
# Set to 2 if the model was generated with CG_mu_deg=2
CG_mu_deg = 2  # <-- CHANGE THIS based on how the data was generated

# Add suffix for CG2 files if applicable
if CG_mu_deg > 1:
    # het3d_str_file = het3d_str + f'_CGmu{CG_mu_deg}'
    # homo_str_file = homo_str + f'_CGmu{CG_mu_deg}'
    het3d_str_file = het3d_str + f'_CG{CG_mu_deg}'
    homo_str_file = homo_str
else:
    het3d_str_file = het3d_str
    homo_str_file = homo_str

# ============================================================================
# Coordinate Transformation Parameters
# (consistent with mesh generation and original notebook)
# ============================================================================
lon0, lat0 = -84, 7      # Reference point (center of projection), from Christos's email
rot = 45                  # Rotation angle in degrees, positive is CCW
x0, y0 = 130e3, 350e3    # Offset for x and y coordinates in meters

# Geographic region for plotting
region = [-87, -84, 8.5, 11.5]         # Full region
region = [-86.5, -84.5, 9, 11]
region = [-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file

region_fault = region   # Fault region for horizontal slices

# 1D reference method: 'step' (step-function, consistent with
# process_velocity_models_hull + scale_shear_modulus_by_1d) or
# 'interp' (linear, consistent with the _interp pair)
mu_1d_ref_method = 'step'

# Save figures to file (1=save, 0=display only)
flag_savefig = 1
# flag_savefig = 0

print(f"Configuration:")
print(f"  Mesh name: {meshname}")
print(f"  Use uneven mesh: {use_uneven_mesh}")
print(f"  CG degree: {CG_mu_deg}")
print(f"  3D model suffix: {het3d_str_file}")
print(f"  Homogeneous model suffix: {homo_str_file}")
print(f"  Coordinate origin: ({lon0}, {lat0})")
print(f"  Save figures: {flag_savefig}")
print(f"  Rotation: {rot}°")
print(f"  Offset: ({x0/1e3:.1f}, {y0/1e3:.1f}) km")

In [ ]:
# ============================================================================
# Load 1D velocity/density model for depth-dependent reference μ
# ============================================================================
veldir = '/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/'

vel1d_ref = pd.read_csv(veldir + 'DeShon2006_1Dmodel.csv', sep=r'\s+', skiprows=1,
                         names=['z', 'vp', 'vs', 'vp_vs_ratio'])
vel1d_ref['z'] = vel1d_ref['z'] * -1 * 1e3   # km → m (negative downward)
vel1d_ref = vel1d_ref[vel1d_ref['z'] <= 0].reset_index(drop=True)

den1d_ref = pd.read_csv(veldir + 'Density_1Dmodel.csv', sep=r'\s+', skiprows=1,
                         names=['z', 'den'])
den1d_ref['z'] = den1d_ref['z'] * -1 * 1e3   # km → m
den1d_ref = den1d_ref[den1d_ref['z'] <= 0].reset_index(drop=True)
den1d_ref['den_si'] = den1d_ref['den'] * 1e3  # g/cm³ → kg/m³

# Merge: compute 1D μ at each velocity depth node
vel1d_merged = vel1d_ref.merge(den1d_ref[['z', 'den_si']], on='z', how='left')
_z_1d  = vel1d_merged['z'].values                                                # m, negative
_mu_1d = (vel1d_merged['den_si'] * (vel1d_merged['vs'] * 1e3)**2 / 1e9).values  # GPa

def mu_1d_at_depth(depth_m):
    """
    Step-function lookup: return 1D reference μ (GPa) at depth_m (metres, negative).
    Consistent with get_layered_1d_value(side='right') in utils.py:
    at an exact layer boundary, the deeper layer value is returned.
    """
    neg_z  = -float(depth_m)        # positive depth
    neg_zs = -_z_1d                 # ascending positive depths [0, 10000, ...]
    idx = int(np.searchsorted(neg_zs, neg_z, side='right')) - 1
    idx = np.clip(idx, 0, len(_mu_1d) - 1)
    return float(_mu_1d[idx])

def mu_1d_ref_at_depths(z_arr_m):
    """
    Vectorized per-DOF 1D μ reference (GPa).
    z_arr_m : array of DOF depths in metres (negative downward).
    Method controlled by mu_1d_ref_method flag in cell 3:
      'step'   → step-function lookup (consistent with process_velocity_models_hull)
      'interp' → linear interpolation (consistent with process_velocity_models_hull_interp)
    """
    z = np.asarray(z_arr_m, dtype=float)
    if mu_1d_ref_method == 'step':
        idx = np.searchsorted(-_z_1d, -z, side='right') - 1
        idx = np.clip(idx, 0, len(_mu_1d) - 1)
        return _mu_1d[idx]
    else:  # 'interp'
        return np.interp(z, _z_1d[::-1], _mu_1d[::-1])

print('1D reference μ model loaded:')
for z_m, mu in zip(_z_1d, _mu_1d):
    print(f'  {-z_m/1e3:6.1f} km:  μ = {mu:.2f} GPa')


In [ ]:
# ============================================================================
# Top Boundary Extraction for Uneven Meshes
# ============================================================================
# For meshes with uneven top boundary (e.g., nicoyaCKden_une_sm), we need to:
# 1. Extract top boundary vertices using FEniCS boundary markers
# 2. Build an interpolator to query z_top(x, y) at any location
# This is used to mask data above the mesh surface in slice plots.
# ============================================================================

def extract_top_boundary_from_mesh(mesh, boundaries, top_id=1):
    """
    Extract top boundary vertices from FEniCS mesh using boundary markers.

    Parameters:
    -----------
    mesh : dolfin.Mesh
        FEniCS mesh object
    boundaries : dolfin.MeshFunction
        Boundary markers (from _facet_region.xml)
    top_id : int
        Boundary ID for top surface (default: 1)

    Returns:
    --------
    top_coords : ndarray
        Nx3 array of (x, y, z) coordinates on top boundary
    """
    import dolfin as dl

    # Get all mesh coordinates
    all_coords = mesh.coordinates()

    # Get facets on the top boundary
    top_facets = []
    for facet in dl.facets(mesh):
        if boundaries[facet] == top_id:
            top_facets.append(facet)

    # Extract unique vertices on top boundary
    top_vertex_indices = set()
    for facet in top_facets:
        for vertex in dl.vertices(facet):
            top_vertex_indices.add(vertex.index())

    # Get coordinates of top boundary vertices
    top_coords = all_coords[list(top_vertex_indices)]

    print(f"Extracted {len(top_coords)} vertices from top boundary (id={top_id})")
    print(f"  x range: [{top_coords[:,0].min()/1e3:.1f}, {top_coords[:,0].max()/1e3:.1f}] km")
    print(f"  y range: [{top_coords[:,1].min()/1e3:.1f}, {top_coords[:,1].max()/1e3:.1f}] km")
    print(f"  z range: [{top_coords[:,2].min()/1e3:.1f}, {top_coords[:,2].max()/1e3:.1f}] km")

    return top_coords


def build_z_top_interpolator(top_coords):
    """
    Build a LinearNDInterpolator for z_top(x, y).

    Parameters:
    -----------
    top_coords : ndarray
        Nx3 array of (x, y, z) coordinates on top boundary

    Returns:
    --------
    z_top_interp : LinearNDInterpolator
        Interpolator that returns z_top given (x, y)
    """
    xy = top_coords[:, :2]
    z = top_coords[:, 2]
    z_top_interp = LinearNDInterpolator(xy, z)
    print(f"Built z_top interpolator from {len(top_coords)} points")
    return z_top_interp


# Initialize z_top_interp as None (for even mesh, no masking needed)
z_top_interp = None

if use_uneven_mesh and DOLFIN_AVAILABLE:
    print("\nLoading mesh for top boundary extraction...")

    # Load mesh
    mesh = dl.Mesh(meshpath + meshname + '.xml')
    boundaries = dl.MeshFunction("size_t", mesh, meshpath + meshname + '_facet_region.xml')

    # Extract top boundary vertices
    top_coords = extract_top_boundary_from_mesh(mesh, boundaries, top_id=1)

    # Build interpolator
    z_top_interp = build_z_top_interpolator(top_coords)

    # Clean up mesh (we only need the interpolator)
    del mesh, boundaries
    print("Mesh unloaded, z_top_interp ready for masking")

elif use_uneven_mesh and not DOLFIN_AVAILABLE:
    print("\nWarning: use_uneven_mesh=True but dolfin not available.")
    print("Top boundary masking will be skipped.")

else:
    print("\nEven mesh detected (use_uneven_mesh=False)")
    print("No top boundary masking will be applied.")

## 2. Load μ from XDMF Files

For CG2 data, we load as a point cloud to preserve all DOF locations (vertices + edge midpoints).

In [ ]:
def load_xdmf_as_pyvista(filepath, cg_degree=1):
    """
    Load XDMF file and convert to PyVista object.
    
    Parameters:
    -----------
    filepath : str
        Path to XDMF file (for CG2+, will also look for corresponding _dofs.h5 file)
    cg_degree : int, default=1
        CG element degree used when saving the function
        - 1: Returns UnstructuredGrid with cell topology (from XDMF)
        - 2+: Returns PolyData point cloud with ALL DOFs (from _dofs.h5 if available)
    
    Returns:
    --------
    grid : pv.UnstructuredGrid or pv.PolyData
        PyVista object with shear modulus data
    """
    import meshio
    import os
    import h5py
    
    if cg_degree == 1:
        # Standard CG1: load from XDMF and create UnstructuredGrid with cell topology
        mesh_data = meshio.read(filepath)
        points = mesh_data.points
        
        print(f"  Loading CG1 from XDMF: {len(points)} points")
        
        cells_data = []
        pv_cell_type = None
        n_points_per_cell = None
        
        for cell_block in mesh_data.cells:
            cell_type = cell_block.type
            cell_data = cell_block.data
            
            if cell_type == 'tetra':
                pv_cell_type = pv.CellType.TETRA
                n_points_per_cell = 4
            elif cell_type == 'hexahedron':
                pv_cell_type = pv.CellType.HEXAHEDRON
                n_points_per_cell = 8
            else:
                continue
                
            for cell in cell_data:
                cells_data.append(n_points_per_cell)
                cells_data.extend(cell)
        
        cells = np.array(cells_data)
        n_cells = len(cells_data) // (n_points_per_cell + 1)
        cell_types = np.full(n_cells, pv_cell_type, dtype=np.uint8)
        
        grid = pv.UnstructuredGrid(cells, cell_types, points)
        
        for name, data in mesh_data.point_data.items():
            grid.point_data[name] = data
        
        print(f"  CG1: Created UnstructuredGrid with {n_cells} cells, {len(points)} vertices")
            
    else:
        # CG2 or higher: try to load ALL DOF points from _dofs.h5 file
        # This file contains coordinates and values at all DOF locations (vertices + edge midpoints)
        
        # Construct the _dofs.h5 filename from the XDMF path
        filepath_base = filepath.replace('.xdmf', '')
        dofs_h5_file = filepath_base + '_dofs.h5'
        
        if os.path.exists(dofs_h5_file):
            # Load from the _dofs.h5 format (contains ALL DOF points)
            print(f"  Loading CG{cg_degree} from _dofs.h5: {dofs_h5_file}")
            
            with h5py.File(dofs_h5_file, 'r') as f:
                dof_coords = f['coordinates'][:]
                dof_values = f['values'][:]
                attr_name = f.attrs['attr_name']
                if isinstance(attr_name, bytes):
                    attr_name = attr_name.decode('utf-8')
                n_dofs = int(f.attrs['n_dofs'])
                n_vertices = int(f.attrs['n_vertices'])
                saved_cg_degree = int(f.attrs['cg_degree'])
            
            # Create PolyData point cloud with all DOF points
            grid = pv.PolyData(dof_coords)
            grid.point_data[attr_name] = dof_values
            
            print(f"  CG{saved_cg_degree}: Created PolyData with {n_dofs} DOF points (vs {n_vertices} mesh vertices)")
            print(f"  Point density ratio: {n_dofs / n_vertices:.1f}x")
            
        else:
            # Fallback: load from XDMF (only has vertex values, not full CG2 DOFs)
            print(f"  WARNING: _dofs.h5 file not found: {dofs_h5_file}")
            print(f"  Falling back to XDMF (only vertex values, not full CG{cg_degree} DOFs)")
            
            mesh_data = meshio.read(filepath)
            points = mesh_data.points
            
            # Use as point cloud
            grid = pv.PolyData(points)
            
            for name, data in mesh_data.point_data.items():
                grid.point_data[name] = data
                
            print(f"  CG{cg_degree} (fallback): Created PolyData with {len(points)} points (vertices only)")
    
    return grid


print("Load function defined")
print("  - CG1: loads from XDMF as UnstructuredGrid")
print("  - CG2+: loads from _dofs.h5 as PolyData point cloud (all DOFs)")
print("         Falls back to XDMF if _dofs.h5 not found")

In [ ]:
# Load mu grids
mu_3d_file = resultpath + f'mu_true_{meshname}{het3d_str_file}.xdmf'

# mu_3d_file = resultpath + 'mu_true_testfact2_nicoyaCKden_une_sm_DeShon3D_ref_4_hull_CG2.xdmf'
# mu_3d_file = resultpath + 'mu_true_testfact2.5_nicoyaCKden_une_sm_DeShon3D_ref_4_hull_CG2.xdmf'

mu_hom_file = resultpath + f'mu_true_{meshname}{homo_str_file}.xdmf'

print(f"Loading 3D mu from: {mu_3d_file}")
mu_3d_grid = load_xdmf_as_pyvista(mu_3d_file, cg_degree=CG_mu_deg)

print(f"\nLoading homogeneous mu from: {mu_hom_file}")
mu_hom_grid = load_xdmf_as_pyvista(mu_hom_file, cg_degree=CG_mu_deg)

# Check field names
print(f"\n3D mu fields: {list(mu_3d_grid.point_data.keys())}")
print(f"Homogeneous mu fields: {list(mu_hom_grid.point_data.keys())}")

In [ ]:
# Print statistics
mu_3d_values = mu_3d_grid['shear modulus']
mu_hom_values = mu_hom_grid['shear modulus']

print("3D Heterogeneous Model Statistics:")
print(f"  Mean: {mu_3d_values.mean():.2f} GPa")
print(f"  Min:  {mu_3d_values.min():.2f} GPa")
print(f"  Max:  {mu_3d_values.max():.2f} GPa")
print(f"  Std:  {mu_3d_values.std():.2f} GPa")

print(f"\nHomogeneous Model Statistics:")
print(f"  Mean: {mu_hom_values.mean():.2f} GPa")
print(f"  Min:  {mu_hom_values.min():.2f} GPa")
print(f"  Max:  {mu_hom_values.max():.2f} GPa")
print(f"  Std:  {mu_hom_values.std():.2f} GPa")

# Reference mu for anomaly calculations
# Option 1: Global mean (commented out)
# mu_ref = mu_3d_values.mean()
# print(f"\nUsing reference mu = {mu_ref:.2f} GPa for anomaly calculations")

# Option 2: Slice-specific mean (computed in anomaly plotting cells)
print(f"\nAnomaly reference: using slice-specific mean at each depth")

## 2.5 Load Plate Interface and Trench

Load the plate interface geometry and trench location for overlaying on plots (consistent with original notebook).

In [ ]:
# Load plate interface geometry
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
plate_mesh = meshio.read("/home/staff/chao/SSEinv/Nicoya/" + mesh_file)
points = plate_mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])

# Convert to geographic coordinates (consistent with mesh generation)
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] / 1e3  # Convert to km

print(f"Plate interface loaded: {len(df_plate)} points")
print(f"  Lon: [{df_plate['lon'].min():.2f}, {df_plate['lon'].max():.2f}]")
print(f"  Lat: [{df_plate['lat'].min():.2f}, {df_plate['lat'].max():.2f}]")
print(f"  Depth: [{df_plate['z'].min():.1f}, {df_plate['z'].max():.1f}] km")

# Load trench line
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)

print(f"\nTrench loaded: {len(trench)} points")
print(f"  Lon: [{trench['lon'].min():.2f}, {trench['lon'].max():.2f}]")
print(f"  Lat: [{trench['lat'].min():.2f}, {trench['lat'].max():.2f}]")

# Create plate interface grid for PyGMT contour plotting
# # Block-median to reduce points, then surface to create grid
# plate_xyz = np.column_stack([df_plate['lon'], df_plate['lat'], df_plate['z']])
# plate_bm = pygmt.blockmedian(data=plate_xyz, region=region, spacing="0.02d")
# plate_grd = pygmt.surface(data=plate_bm, region=region, spacing="0.02d", tension=0.25)

plate_grd = pygmt.xyz2grd(
    x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'],
    region=region_fault, spacing=(0.05, 0.05)
)

print("\nPlate interface grid created for contour plotting")

## 3. Slice Extraction Functions

**Unified approach that matches the original notebook philosophy:**

- **CG1 (UnstructuredGrid)**: Uses PyVista's `.slice()` method for exact plane-mesh intersection
- **CG2 (PolyData point cloud)**: Uses tolerance-based filtering to extract points near the slice plane

Both approaches then feed into the same `griddata` interpolation to create regular grids for plotting.

In [ ]:
def extract_horizontal_slice(grid, z_depth, tolerance=1000, field_name='shear modulus',
                            z_top_interp=None):
    """
    Extract a horizontal slice from grid or point cloud.

    - For UnstructuredGrid (CG1): uses PyVista's .slice() for exact plane intersection
    - For PolyData (CG2 point cloud): uses tolerance-based filtering

    Parameters:
    -----------
    grid : pv.PolyData or pv.UnstructuredGrid
        Grid or point cloud with field values
    z_depth : float
        Z coordinate of slice (meters, negative for depth)
    tolerance : float
        Half-thickness for point cloud filtering (meters)
        Ignored for UnstructuredGrid (uses exact slice)
    field_name : str
        Name of the field to extract
    z_top_interp : LinearNDInterpolator or None
        Interpolator for top boundary z(x,y). If provided, points above
        top boundary are masked (set to NaN). Use for uneven mesh.

    Returns:
    --------
    x, y, z_pts : ndarray
        X, Y, Z coordinates of extracted points (meters)
    values : ndarray
        Field values at those points (NaN for points above top boundary)
    """
    if isinstance(grid, pv.UnstructuredGrid):
        # CG1: Use PyVista's slice method (exact plane intersection)
        slice_obj = grid.slice(normal='z', origin=[0, 0, z_depth])
        if len(slice_obj.points) == 0:
            return np.array([]), np.array([]), np.array([]), np.array([])
        x = slice_obj.points[:, 0]
        y = slice_obj.points[:, 1]
        z_pts = slice_obj.points[:, 2]
        values = slice_obj[field_name].copy()
    else:
        # CG2 PolyData: Use tolerance-based filtering
        points = grid.points
        z = points[:, 2]
        mask = np.abs(z - z_depth) < tolerance
        if mask.sum() == 0:
            return np.array([]), np.array([]), np.array([]), np.array([])
        x = points[mask, 0]
        y = points[mask, 1]
        z_pts = z[mask]
        values = grid[field_name][mask].copy()

    # Apply top boundary masking if interpolator provided
    if z_top_interp is not None:
        z_top_at_points = z_top_interp(np.column_stack([x, y]))
        above_top = z_depth > z_top_at_points  # z_depth is the slice level
        values[above_top] = np.nan

    return x, y, z_pts, values


def extract_vertical_slice_dip(grid, y_const, tolerance=2000, field_name='shear modulus',
                               z_top_interp=None):
    """
    Extract a vertical slice along dip (constant y).

    - For UnstructuredGrid (CG1): uses PyVista's .slice() for exact plane intersection
    - For PolyData (CG2 point cloud): uses tolerance-based filtering

    Parameters:
    -----------
    grid : pv.PolyData or pv.UnstructuredGrid
        Grid or point cloud with field values
    y_const : float
        Y coordinate of slice (meters)
    tolerance : float
        Half-thickness for point cloud filtering (meters)
    field_name : str
        Name of the field to extract
    z_top_interp : LinearNDInterpolator or None
        Interpolator for top boundary z(x,y). If provided, points above
        top boundary are masked (set to NaN). Use for uneven mesh.

    Returns:
    --------
    x, z : ndarray
        X and Z coordinates of extracted points (meters)
    values : ndarray
        Field values at those points (NaN for points above top boundary)
    """
    if isinstance(grid, pv.UnstructuredGrid):
        # CG1: Use PyVista's slice method
        slice_obj = grid.slice(normal='y', origin=[0, y_const, 0])
        if len(slice_obj.points) == 0:
            return np.array([]), np.array([]), np.array([])
        x = slice_obj.points[:, 0]
        z = slice_obj.points[:, 2]
        values = slice_obj[field_name].copy()
    else:
        # CG2 PolyData: Use tolerance-based filtering
        points = grid.points
        y = points[:, 1]
        mask = np.abs(y - y_const) < tolerance
        if mask.sum() == 0:
            return np.array([]), np.array([]), np.array([])
        x = points[mask, 0]
        z = points[mask, 2]
        values = grid[field_name][mask].copy()

    # Apply top boundary masking if interpolator provided
    if z_top_interp is not None:
        # For vertical slice at constant y, query z_top at (x, y_const)
        xy_query = np.column_stack([x, np.full_like(x, y_const)])
        z_top_at_points = z_top_interp(xy_query)
        above_top = z > z_top_at_points
        values[above_top] = np.nan

    return x, z, values


def extract_vertical_slice_strike(grid, x_const, tolerance=2000, field_name='shear modulus',
                                  z_top_interp=None):
    """
    Extract a vertical slice along strike (constant x).

    - For UnstructuredGrid (CG1): uses PyVista's .slice() for exact plane intersection
    - For PolyData (CG2 point cloud): uses tolerance-based filtering

    Parameters:
    -----------
    grid : pv.PolyData or pv.UnstructuredGrid
        Grid or point cloud with field values
    x_const : float
        X coordinate of slice (meters)
    tolerance : float
        Half-thickness for point cloud filtering (meters)
    field_name : str
        Name of the field to extract
    z_top_interp : LinearNDInterpolator or None
        Interpolator for top boundary z(x,y). If provided, points above
        top boundary are masked (set to NaN). Use for uneven mesh.

    Returns:
    --------
    y, z : ndarray
        Y and Z coordinates of extracted points (meters)
    values : ndarray
        Field values at those points (NaN for points above top boundary)
    """
    if isinstance(grid, pv.UnstructuredGrid):
        # CG1: Use PyVista's slice method
        slice_obj = grid.slice(normal='x', origin=[x_const, 0, 0])
        if len(slice_obj.points) == 0:
            return np.array([]), np.array([]), np.array([])
        y = slice_obj.points[:, 1]
        z = slice_obj.points[:, 2]
        values = slice_obj[field_name].copy()
    else:
        # CG2 PolyData: Use tolerance-based filtering
        points = grid.points
        x = points[:, 0]
        mask = np.abs(x - x_const) < tolerance
        if mask.sum() == 0:
            return np.array([]), np.array([]), np.array([])
        y = points[mask, 1]
        z = points[mask, 2]
        values = grid[field_name][mask].copy()

    # Apply top boundary masking if interpolator provided
    if z_top_interp is not None:
        # For vertical slice at constant x, query z_top at (x_const, y)
        xy_query = np.column_stack([np.full_like(y, x_const), y])
        z_top_at_points = z_top_interp(xy_query)
        above_top = z > z_top_at_points
        values[above_top] = np.nan

    return y, z, values


def interpolate_to_grid(x, y, values, grid_resolution=100, method='linear'):
    """
    Interpolate scattered points to a regular grid.

    Parameters:
    -----------
    x, y : ndarray
        Coordinates of scattered points
    values : ndarray
        Values at scattered points
    grid_resolution : int
        Number of grid points in each direction
    method : str
        Interpolation method ('linear', 'nearest', 'cubic')

    Returns:
    --------
    xi, yi : ndarray
        Regular grid coordinates
    zi : ndarray
        Interpolated values on the grid
    """
    if len(x) == 0:
        return np.array([]), np.array([]), np.array([])

    # Create regular grid
    xi = np.linspace(x.min(), x.max(), grid_resolution)
    yi = np.linspace(y.min(), y.max(), grid_resolution)
    xi, yi = np.meshgrid(xi, yi)

    # Interpolate
    zi = griddata((x, y), values, (xi, yi), method=method)

    return xi, yi, zi


def get_top_boundary_line_dip(y_const, x_range, z_top_interp, n_points=200):
    """
    Get the top boundary line for a vertical dip slice (constant y).

    Parameters:
    -----------
    y_const : float
        Y coordinate of the slice (meters)
    x_range : tuple
        (x_min, x_max) in km
    z_top_interp : LinearNDInterpolator
        Interpolator for z_top(x, y)
    n_points : int
        Number of points for the line

    Returns:
    --------
    x_line : ndarray
        X coordinates (km)
    z_top_line : ndarray
        Z coordinates of top boundary (km)
    """
    x_min_m, x_max_m = x_range[0] * 1e3, x_range[1] * 1e3
    x_line_m = np.linspace(x_min_m, x_max_m, n_points)
    xy_query = np.column_stack([x_line_m, np.full_like(x_line_m, y_const)])
    z_top_m = z_top_interp(xy_query)
    return x_line_m / 1e3, z_top_m / 1e3


def get_top_boundary_line_strike(x_const, y_range, z_top_interp, n_points=200):
    """
    Get the top boundary line for a vertical strike slice (constant x).

    Parameters:
    -----------
    x_const : float
        X coordinate of the slice (meters)
    y_range : tuple
        (y_min, y_max) in km
    z_top_interp : LinearNDInterpolator
        Interpolator for z_top(x, y)
    n_points : int
        Number of points for the line

    Returns:
    --------
    y_line : ndarray
        Y coordinates (km)
    z_top_line : ndarray
        Z coordinates of top boundary (km)
    """
    y_min_m, y_max_m = y_range[0] * 1e3, y_range[1] * 1e3
    y_line_m = np.linspace(y_min_m, y_max_m, n_points)
    xy_query = np.column_stack([np.full_like(y_line_m, x_const), y_line_m])
    z_top_m = z_top_interp(xy_query)
    return y_line_m / 1e3, z_top_m / 1e3


print("Slice extraction functions defined")
print("  - CG1 (UnstructuredGrid): uses PyVista .slice() for exact plane intersection")
print("  - CG2 (PolyData): uses tolerance-based filtering to preserve all DOF points")
print("  - z_top_interp parameter: masks points above top boundary (for uneven mesh)")
print("  - get_top_boundary_line_dip/strike: helper functions for plotting boundary line")

## 4. Horizontal Slices at Different Depths

In [ ]:
# Get domain bounds
bounds = mu_3d_grid.bounds
print(f"Domain bounds:")
print(f"  X: {bounds[0]/1e3:.1f} to {bounds[1]/1e3:.1f} km")
print(f"  Y: {bounds[2]/1e3:.1f} to {bounds[3]/1e3:.1f} km")
print(f"  Z: {bounds[4]/1e3:.1f} to {bounds[5]/1e3:.1f} km")

In [ ]:
# Horizontal slice parameters
# depth_levels_km = [10, 20, 30, 40, 50]  # km (positive values for plotting convenience)
depth_levels_km = [20, 30, 40, 50]  # km (positive values for plotting convenience)
depth_levels_m = [-d * 1e3 for d in depth_levels_km]  # meters (negative)

# Slice tolerance (half-thickness)
slice_tolerance = 1000  # meters

# Grid resolution for interpolation
grid_res = 1000  # total grid points in each direction

print(f"Extracting horizontal slices at depths: {depth_levels_km} km")
print(f"Slice tolerance: {slice_tolerance/1e3:.1f} km")

In [ ]:
# Extract and plot horizontal slices using matplotlib
# fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

# Color scale
vmin, vmax = mu_3d_values.min(), mu_3d_values.max()
# vmin, vmax = 5, 160
print(f"Color scale for horizontal slices: vmin={vmin:.1f} GPa, vmax={vmax:.1f} GPa")

# cmap = plt.cm.get_cmap('cmc.roma', 100) 
cmap = 'cmc.roma'

for i, (depth_km, depth_m) in enumerate(zip(depth_levels_km, depth_levels_m)):
    if i >= len(axes):
        break
        
    ax = axes[i]
    
    # Extract slice
    x, y, z_pts, mu_vals = extract_horizontal_slice(mu_3d_grid, depth_m, 
                                                    tolerance=slice_tolerance)
    
    if len(x) < 10:
        ax.set_title(f'Depth = {depth_km} km\n(insufficient data)')
        ax.axis('off')
        continue
    
    # Interpolate to regular grid
    xi, yi, zi = interpolate_to_grid(x, y, mu_vals, grid_resolution=grid_res)
    
    # Plot
    im = ax.pcolormesh(xi/1e3, yi/1e3, zi, cmap=cmap, 
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_title(f'Depth = {depth_km} km\n({len(x)} points)')
    ax.set_xlim([-100, 100])
    ax.set_ylim([-100, 100])
    ax.set_aspect('equal')
    
# Remove unused axes
for j in range(i+1, len(axes)):
    axes[j].axis('off')

# Colorbar
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Shear Modulus (GPa)')

plt.suptitle(f'Horizontal Slices - 3D Heterogeneous Model (CG{CG_mu_deg})', fontsize=14)
plt.tight_layout(rect=[0, 0, 0.88, 0.96])
plt.show()

In [ ]:
# # Plot μ anomaly horizontal slices
# fig, axes = plt.subplots(2, 2, figsize=(10, 10))
# axes = axes.flatten()

# # Anomaly color scale (symmetric)
# anom_max = 100  # percent

# for i, (depth_km, depth_m) in enumerate(zip(depth_levels_km, depth_levels_m)):
#     if i >= len(axes):
#         break
        
#     ax = axes[i]
    
#     # Extract slice
#     x, y, z_pts, mu_vals = extract_horizontal_slice(mu_3d_grid, depth_m, 
#                                                     tolerance=slice_tolerance)
    
#     if len(x) < 10:
#         ax.set_title(f'Depth = {depth_km} km\n(insufficient data)')
#         ax.axis('off')
#         continue
    
#     # Compute slice-specific reference mu (mean of this slice)
#     mu_ref_slice = mu_vals.mean()
    
#     # Compute anomaly relative to slice mean
#     mu_anomaly = (mu_vals - mu_ref_slice) / mu_ref_slice * 100
#     print(mu_anomaly.min(), mu_anomaly.max())

#     # Interpolate to regular grid
#     xi, yi, zi = interpolate_to_grid(x, y, mu_anomaly, grid_resolution=grid_res)
    
#     # Plot
#     im = ax.pcolormesh(xi/1e3, yi/1e3, zi, cmap='cmc.roma', 
#                        vmin=-anom_max, vmax=anom_max, shading='auto')
#     ax.set_xlabel('X (km)')
#     ax.set_ylabel('Y (km)')
#     ax.set_title(f'Depth = {depth_km} km\n(ref={mu_ref_slice:.1f} GPa, {len(x)} pts)')
#     ax.set_xlim([-100, 100])
#     ax.set_ylim([-100, 100])
#     ax.set_aspect('equal')

# # Colorbar
# fig.subplots_adjust(right=0.88)
# cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
# cbar = fig.colorbar(im, cax=cbar_ax)
# cbar.set_label('μ Anomaly (%)')

# plt.suptitle(f'Horizontal Slices - μ Anomaly (slice-specific ref, CG{CG_mu_deg})', fontsize=14)
# plt.tight_layout(rect=[0, 0, 0.88, 0.96])
# plt.show()

## 4.1. Compare CG1 vs CG2 Point Density

This section demonstrates the difference in point density between CG1 and CG2.

In [ ]:
# %%
from matplotlib.patches import Polygon
import cartopy.crs as ccrs

# read in the 3D velocity model
veldir = "/home/staff/chao/SSEinv/Nicoya/DeShon_2006GJI/"
vel3dfile = "DeShon2006_3Dmodel.csv"
vel3d = pd.read_csv(veldir + vel3dfile, sep=",")
print(f'X range: {vel3d["lon"].min():.2f} to {vel3d["lon"].max():.2f} degrees')
print(f'Y range: {vel3d["lat"].min():.2f} to {vel3d["lat"].max():.2f} degrees')
print(f'X span: {vel3d["lon"].max() - vel3d["lon"].min():.2f} degrees')
print(f'Y span: {vel3d["lat"].max() - vel3d["lat"].min():.2f} degrees')

# convert to relative locations in meters, and then rotate, then offset, align with the local coordinate system of the mesh
x_rot, y_rot = utp.LL2ckmd(vel3d['lon'], vel3d['lat'], lon0, lat0, rot)
vel3d['x'], vel3d['y'] = x_rot - x0, y_rot - y0   # offset to match the mesh coordinates
vel3d['z'] = vel3d['z'] * -1 * 1e3  # negative depth means downward
vel3d = vel3d[(vel3d['z'] <= 0)].reset_index(drop=True)  # ignore everything above the ground
# print(vel3d.shape)
# print(vel3d.head())
# print(f'X range: {vel3d["x"].min():.2f} to {vel3d["x"].max():.2f} meters')
# print(f'Y range: {vel3d["y"].min():.2f} to {vel3d["y"].max():.2f} meters')
print(f'Z range: {vel3d["z"].min():.2f} to {vel3d["z"].max():.2f} meters')
# print(f'X span: {vel3d["x"].max() - vel3d["x"].min():.2f} meters')
# print(f'Y span: {vel3d["y"].max() - vel3d["y"].min():.2f} meters')

lon_min, lon_max = vel3d['lon'].min(), vel3d['lon'].max()
lat_min, lat_max = vel3d['lat'].min(), vel3d['lat'].max()

# Create polygon for lat/lon range
poly_latlon = Polygon([
    [lon_min, lat_min],
    [lon_max, lat_min],
    [lon_max, lat_max],
    [lon_min, lat_max]
], closed=True, edgecolor='red', facecolor='none', linewidth=2, transform=ccrs.PlateCarree())

# Transform the CORNERS of the original lat/lon bounding box
corners_lon = [lon_min, lon_max, lon_max, lon_min]
corners_lat = [lat_min, lat_min, lat_max, lat_max]
corners_x_rot, corners_y_rot = utp.LL2ckmd(np.array(corners_lon), np.array(corners_lat), lon0, lat0, rot)
corners_x = (corners_x_rot - x0)/1e3
corners_y = (corners_y_rot - y0)/1e3

poly_xy = [
    [corners_x[0], corners_y[0]],
    [corners_x[1], corners_y[1]],
    [corners_x[2], corners_y[2]],
    [corners_x[3], corners_y[3]],
]


In [ ]:
# Load both CG1 and CG2 versions if available for comparison
import os

try:
    # mu_cg1_file = resultpath + f'mu_true_{meshname}{het3d_str}.xdmf'  # CG1, old function
    mu_cg1_file = resultpath + f'mu_true_{meshname}{het3d_str}.xdmf'    # CG1, convex hull function
    mu_cg2_file = resultpath + f'mu_true_{meshname}{het3d_str}_CG2.xdmf'
    mu_cg2_dofs_file = resultpath + f'mu_true_{meshname}{het3d_str}_CG2_dofs.h5'
    
    print("Attempting to load both CG1 and CG2 for comparison...")
    print(f"CG1 file: {mu_cg1_file}")
    print(f"CG2 XDMF file: {mu_cg2_file}")
    print(f"CG2 DOFs file: {mu_cg2_dofs_file}")
    print(f"  _dofs.h5 exists: {os.path.exists(mu_cg2_dofs_file)}")
    print()
    
    mu_cg1 = load_xdmf_as_pyvista(mu_cg1_file, cg_degree=1)
    print()
    mu_cg2 = load_xdmf_as_pyvista(mu_cg2_file, cg_degree=2)
    
    print(f"\n=== Comparison Summary ===")
    print(f"CG1: {mu_cg1.n_points} points (mesh vertices)")
    print(f"CG2: {mu_cg2.n_points} points (DOFs = vertices + edge midpoints)")
    
    if mu_cg2.n_points > mu_cg1.n_points:
        ratio = mu_cg2.n_points / mu_cg1.n_points
        print(f"Ratio: {ratio:.2f}x more points with CG2")
        print("✓ CG2 data loaded correctly with all DOF points!")
    else:
        print("⚠ CG2 has same point count as CG1 - _dofs.h5 may be missing")
        print("  Run the forward model with CG_mu_deg=2 to generate the _dofs.h5 file")
    
    comparison_available = True
except Exception as e:
    print(f"Could not load both versions: {e}")
    print("Comparison not available.")
    comparison_available = False

In [ ]:
if comparison_available:
    # Compare slice density
    depth_m = -40e3  # 30 km depth
    tolerance = 1000    # meters

    x1, y1, v1 = extract_horizontal_slice(mu_cg1, depth_m, tolerance=tolerance)
    x2, y2, v2 = extract_horizontal_slice(mu_cg2, depth_m, tolerance=tolerance)
    
    print(f"At depth = {depth_m/1e3:.0f} km:")
    print(f"  CG1: {len(x1)} points in slice")
    print(f"  CG2: {len(x2)} points in slice")
    print(f"  Ratio: {len(x2)/len(x1):.2f}x")
    
    # Visual comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # CG1
    ax = axes[0]
    ax.scatter(x1/1e3, y1/1e3, c=v1, s=2, cmap='viridis')
    poly1 = Polygon(poly_xy, closed=True, fill=False, edgecolor='red', linewidth=2,
                    label='Original lat/lon extent')
    ax.add_patch(poly1)
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_title(f'CG1 - {len(x1)} points')
    ax.set_aspect('equal')
    
    # CG2
    ax = axes[1]
    sc = ax.scatter(x2/1e3, y2/1e3, c=v2, s=2, cmap='viridis')
    poly2 = Polygon(poly_xy, closed=True, fill=False, edgecolor='red', linewidth=2,
                label='Original lat/lon extent')
    ax.add_patch(poly2)
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_title(f'CG2 - {len(x2)} points')
    ax.set_aspect('equal')
    
    plt.colorbar(sc, ax=axes, label='μ (GPa)', shrink=0.8)
    plt.suptitle(f'Point Density Comparison: CG1 vs CG2 at {depth_m/1e3:.0f} km depth')
    # plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================================
# CRITICAL COMPARISON: CG1 slice vs CG1 tolerance vs CG2 tolerance
# ============================================================================
# This answers: Does smoothness come from higher CG order, or from the method?

def extract_horizontal_slice_tolerance(grid, z_depth, tolerance=1000, field_name='shear modulus'):
    """
    Extract horizontal slice using tolerance-based filtering (for ANY grid type).
    This ignores cell topology and just filters points by z-coordinate.
    """
    if isinstance(grid, pv.UnstructuredGrid):
        points = grid.points
    else:
        points = grid.points
    
    z = points[:, 2]
    mask = np.abs(z - z_depth) < tolerance
    
    if mask.sum() == 0:
        return np.array([]), np.array([]), np.array([])
    
    x = points[mask, 0]
    y = points[mask, 1]
    values = grid[field_name][mask]
    
    return x, y, values


def extract_horizontal_slice_pyvista(grid, z_depth, field_name='shear modulus'):
    """
    Extract horizontal slice using PyVista's .slice() method.
    Only works for UnstructuredGrid with cell topology.
    """
    if not isinstance(grid, pv.UnstructuredGrid):
        raise TypeError("PyVista slice requires UnstructuredGrid with cell topology")
    
    slice_obj = grid.slice(normal='z', origin=[0, 0, z_depth])
    
    if len(slice_obj.points) == 0:
        return np.array([]), np.array([]), np.array([])
    
    x = slice_obj.points[:, 0]
    y = slice_obj.points[:, 1]
    values = slice_obj[field_name]
    
    return x, y, values


if comparison_available:
    print("=" * 70)
    print("COMPARISON: Does smoothness come from CG order or extraction method?")
    print("=" * 70)

    # Compare slice density
    depth_m = -40e3  # 30 km depth
    tolerance = 1000    # meters

    # Method 1: CG1 with PyVista .slice() (creates new interpolated points)
    x1_slice, y1_slice, v1_slice = extract_horizontal_slice_pyvista(mu_cg1, depth_m)
    
    # Method 2: CG1 with tolerance filtering (uses actual vertex points)
    x1_tol, y1_tol, v1_tol = extract_horizontal_slice_tolerance(mu_cg1, depth_m, tolerance)
    
    # Method 3: CG2 with tolerance filtering (uses actual DOF points)
    x2_tol, y2_tol, v2_tol = extract_horizontal_slice_tolerance(mu_cg2, depth_m, tolerance)
    
    print(f"\nAt depth = {depth_m/1e3:.0f} km (tolerance = {tolerance/1e3} km):")
    print(f"  CG1 + slice():    {len(x1_slice):6d} points (interpolated at plane intersection)")
    print(f"  CG1 + tolerance:  {len(x1_tol):6d} points (actual vertices in slab)")
    print(f"  CG2 + tolerance:  {len(x2_tol):6d} points (actual DOFs in slab)")
    print(f"\nRatios:")
    print(f"  CG1 slice / CG1 tolerance: {len(x1_slice)/max(len(x1_tol),1):.2f}x")
    print(f"  CG2 tolerance / CG1 slice: {len(x2_tol)/max(len(x1_slice),1):.2f}x")
    print(f"  CG2 tolerance / CG1 tolerance: {len(x2_tol)/max(len(x1_tol),1):.2f}x")

    
# ============================================================================
# Visual comparison: 3 methods side by side
# ============================================================================

if comparison_available:
    # Interpolate all three to the same regular grid for fair comparison
    grid_res_compare = 1000
    xlim, ylim = [-100, 100], [-100, 100]  # km
    # xlim, ylim = [-250, 250], [-250, 250]  # km
    # xlim, ylim = [-500, 500], [-500, 500]  # km
    # xlim, ylim = [-1000, 1000], [-1000, 1000]  # km
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=150)
    vmin, vmax = 20, 160  # GPa
    
    # --- Panel 1: CG1 + slice() ---
    ax = axes[0]
    if len(x1_slice) > 10:
        xi, yi, zi = interpolate_to_grid(x1_slice, y1_slice, v1_slice, grid_resolution=grid_res_compare)
        im = ax.pcolormesh(xi/1e3, yi/1e3, zi, cmap='cmc.roma', vmin=vmin, vmax=vmax, shading='auto')
        poly1 = Polygon(poly_xy, closed=True, fill=False, edgecolor='red', linewidth=2,
                label='Original lat/lon extent')
        ax.add_patch(poly1)
        ax.set_title(f'CG1 + slice()\n{len(x1_slice)} pts (interpolated)', fontsize=11)
    else:
        ax.set_title('CG1 + slice()\n(insufficient data)')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    
    # --- Panel 2: CG1 + tolerance ---
    ax = axes[1]
    if len(x1_tol) > 10:
        xi, yi, zi = interpolate_to_grid(x1_tol, y1_tol, v1_tol, grid_resolution=grid_res_compare)
        im = ax.pcolormesh(xi/1e3, yi/1e3, zi, cmap='cmc.roma', vmin=vmin, vmax=vmax, shading='auto')
        poly1 = Polygon(poly_xy, closed=True, fill=False, edgecolor='red', linewidth=2,
                label='Original lat/lon extent')
        ax.add_patch(poly1)
        ax.set_title(f'CG1 + tolerance\n{len(x1_tol)} pts (vertices only)', fontsize=11)
    else:
        ax.set_title('CG1 + tolerance\n(insufficient data)')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    
    # --- Panel 3: CG2 + tolerance ---
    ax = axes[2]
    if len(x2_tol) > 10:
        xi, yi, zi = interpolate_to_grid(x2_tol, y2_tol, v2_tol, grid_resolution=grid_res_compare)
        im = ax.pcolormesh(xi/1e3, yi/1e3, zi, cmap='cmc.roma', vmin=vmin, vmax=vmax, shading='auto')
        poly1 = Polygon(poly_xy, closed=True, fill=False, edgecolor='red', linewidth=2,
                label='Original lat/lon extent')
        ax.add_patch(poly1)
        ax.set_title(f'CG2 + tolerance\n{len(x2_tol)} pts (vertices + edge midpoints)', fontsize=11)
    else:
        ax.set_title('CG2 + tolerance\n(insufficient data)')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    
    # Colorbar
    fig.subplots_adjust(right=0.92)
    cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label('Shear Modulus (GPa)')
    
    plt.suptitle(f'Method Comparison at {depth_m/1e3:.0f} km depth (tolerance = {tolerance/1e3} km)', fontsize=14)
    plt.tight_layout(rect=[0, 0, 0.92, 0.95])
    plt.show()
    
    print("\n" + "=" * 70)
    print("INTERPRETATION:")
    print("=" * 70)
    print("• If CG1+slice and CG1+tolerance look similar → method doesn't matter much")
    print("• If CG2+tolerance is smoother than both CG1 methods → CG order matters")
    print("• If CG1+slice is smoother than CG1+tolerance → slice interpolation helps")
    print("=" * 70)

In [ ]:
# ============================================================================
# Raw point distribution (before griddata interpolation)
# ============================================================================
# This shows the actual sampling density differences

if comparison_available:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=150)
    vmin, vmax = 20, 160
    xlim, ylim = [-100, 100], [-100, 100]
    
    # --- Panel 1: CG1 + slice() ---
    ax = axes[0]
    if len(x1_slice) > 10:
        sc = ax.scatter(x1_slice/1e3, y1_slice/1e3, c=v1_slice, s=1, cmap='cmc.roma', vmin=vmin, vmax=vmax)
        ax.set_title(f'CG1 + slice()\n{len(x1_slice)} pts', fontsize=11)
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    
    # --- Panel 2: CG1 + tolerance ---
    ax = axes[1]
    if len(x1_tol) > 10:
        sc = ax.scatter(x1_tol/1e3, y1_tol/1e3, c=v1_tol, s=1, cmap='cmc.roma', vmin=vmin, vmax=vmax)
        ax.set_title(f'CG1 + tolerance\n{len(x1_tol)} pts', fontsize=11)
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    
    # --- Panel 3: CG2 + tolerance ---
    ax = axes[2]
    if len(x2_tol) > 10:
        sc = ax.scatter(x2_tol/1e3, y2_tol/1e3, c=v2_tol, s=1, cmap='cmc.roma', vmin=vmin, vmax=vmax)
        ax.set_title(f'CG2 + tolerance\n{len(x2_tol)} pts', fontsize=11)
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect('equal')
    
    # Colorbar
    fig.subplots_adjust(right=0.92)
    cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
    cbar = fig.colorbar(sc, cax=cbar_ax)
    cbar.set_label('Shear Modulus (GPa)')
    
    plt.suptitle(f'Raw Point Distribution at 30 km depth (before griddata)', fontsize=14)
    plt.tight_layout(rect=[0, 0, 0.92, 0.95])
    plt.show()
    
    print("\nNote: CG1+tolerance has sparse, irregular coverage because few vertices")
    print("      happen to fall within the tolerance slab at this depth.")

## 5. Vertical Slices Along Dip (Constant Y)

In [ ]:
# Vertical slice parameters (along dip)
# y_dip_km = [-50, -25, 0, 25, 50]  # km from origin
y_dip_km = [-10, 10, 30, 50]  # km from origin
y_dip_m = [y * 1e3 for y in y_dip_km]  # meters

x_min, x_max = -80, 60  # km
z_min, z_max = -60, 0    # km

# Slice tolerance
slice_tolerance_v = 1500  # meters

print(f"Extracting vertical slices at Y = {y_dip_km} km")
# Extract and plot vertical slices along dip
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

# Color scale
vmin, vmax = mu_3d_values.min(), mu_3d_values.max()
# vmin, vmax = 5, 160

# cmap = plt.get_cmap('cmc.roma', 20) 
cmap = 'cmc.roma'

for i, (y_km, y_m) in enumerate(zip(y_dip_km, y_dip_m)):
    if i >= len(axes):
        break
        
    ax = axes[i]
    
    # Extract slice
    x, z, mu_vals = extract_vertical_slice_dip(mu_3d_grid, y_m, 
                                                tolerance=slice_tolerance_v)
    
    if len(x) < 10:
        ax.set_title(f'Y = {y_km} km\n(insufficient data)')
        ax.axis('off')
        continue
    
    # Interpolate to regular grid
    xi, zi_grid, mu_interp = interpolate_to_grid(x, z, mu_vals, grid_resolution=grid_res)
    
    # Plot
    im = ax.pcolormesh(xi/1e3, zi_grid/1e3, mu_interp, cmap=cmap, 
                       vmin=vmin, vmax=vmax, shading='auto')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Z (km)')
    ax.set_title(f'Y = {y_km} km\n({len(x)} points)')
    ax.set_xlim([x_min, x_max])
    ax.set_ylim([z_min, z_max])
    ax.set_aspect('equal')
    # ax.invert_yaxis()  # Depth increases downward
    
# Remove unused axes
for j in range(i+1, len(axes)):
    axes[j].axis('off')

# Colorbar
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Shear Modulus (GPa)')

plt.suptitle(f'Vertical Slices Along Dip (constant Y) - CG{CG_mu_deg}', fontsize=14)
plt.tight_layout(rect=[0, 0, 0.88, 0.96])
plt.show()

In [ ]:
# # Vertical slices along dip - μ anomaly
# fig, axes = plt.subplots(2, 2, figsize=(15, 10))
# axes = axes.flatten()

# anom_max = 100  # percent

# for i, (y_km, y_m) in enumerate(zip(y_dip_km, y_dip_m)):
#     if i >= len(axes):
#         break
        
#     ax = axes[i]
    
#     # Extract slice
#     x, z, mu_vals = extract_vertical_slice_dip(mu_3d_grid, y_m, 
#                                                 tolerance=slice_tolerance_v)
    
#     if len(x) < 10:
#         ax.set_title(f'Y = {y_km} km\n(insufficient data)')
#         ax.axis('off')
#         continue
    
#     # Compute slice-specific reference mu (mean of this slice)
#     mu_ref_slice = mu_vals.mean()
    
#     # Compute anomaly relative to slice mean
#     mu_anomaly = (mu_vals - mu_ref_slice) / mu_ref_slice * 100
    
#     # Interpolate to regular grid
#     xi, zi_grid, anom_interp = interpolate_to_grid(x, z, mu_anomaly, grid_resolution=grid_res)
    
#     # Plot
#     im = ax.pcolormesh(xi/1e3, zi_grid/1e3, anom_interp, cmap='cmc.roma', 
#                        vmin=-anom_max, vmax=anom_max, shading='auto')
#     ax.set_xlabel('X (km)')
#     ax.set_ylabel('Z (km)')
#     ax.set_title(f'Y = {y_km} km\n(ref={mu_ref_slice:.1f} GPa, {len(x)} pts)')
#     ax.set_xlim([x_min, x_max])
#     ax.set_ylim([z_min, z_max])
#     ax.set_aspect('equal')

# # Colorbar
# fig.subplots_adjust(right=0.88)
# cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
# cbar = fig.colorbar(im, cax=cbar_ax)
# cbar.set_label('μ Anomaly (%)')

# plt.suptitle(f'Vertical Slices Along Dip - μ Anomaly (slice-specific ref, CG{CG_mu_deg})', fontsize=14)
# plt.tight_layout(rect=[0, 0, 0.88, 0.96])
# plt.show()

## 6. Vertical Slices Along Strike (Constant X)

In [ ]:
# # Vertical slice parameters (along strike)
# x_strike_km = [-30, 0, 30, 60]  # km from origin (4 slices for 2x2 grid)
# x_strike_m = [x * 1e3 for x in x_strike_km]  # meters

# y_min, y_max = -60, 80  # km
# z_min_s, z_max_s = -60, 0  # km

# # Slice tolerance
# slice_tolerance_s = 2500  # meters

# print(f"Extracting vertical slices at X = {x_strike_km} km")

In [ ]:
# # Extract and plot vertical slices along strike - absolute μ
# fig, axes = plt.subplots(2, 2, figsize=(15, 10))
# axes = axes.flatten()

# # Color scale
# vmin, vmax = 5, 160

# cmap = 'cmc.roma'

# for i, (x_km, x_m) in enumerate(zip(x_strike_km, x_strike_m)):
#     if i >= len(axes):
#         break
        
#     ax = axes[i]
    
#     # Extract slice
#     y, z, mu_vals = extract_vertical_slice_strike(mu_3d_grid, x_m, 
#                                                    tolerance=slice_tolerance_s)
    
#     if len(y) < 10:
#         ax.set_title(f'X = {x_km} km\n(insufficient data)')
#         ax.axis('off')
#         continue
    
#     # Interpolate to regular grid
#     yi, zi_grid, mu_interp = interpolate_to_grid(y, z, mu_vals, grid_resolution=grid_res)
    
#     # Plot
#     im = ax.pcolormesh(yi/1e3, zi_grid/1e3, mu_interp, cmap=cmap, 
#                        vmin=vmin, vmax=vmax, shading='auto')
#     ax.set_xlabel('Y (km)')
#     ax.set_ylabel('Z (km)')
#     ax.set_title(f'X = {x_km} km\n({len(y)} points)')
#     ax.set_xlim([y_min, y_max])
#     ax.set_ylim([z_min_s, z_max_s])
#     ax.set_aspect('equal')
    
# # Remove unused axes
# for j in range(i+1, len(axes)):
#     axes[j].axis('off')

# # Colorbar
# fig.subplots_adjust(right=0.88)
# cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
# cbar = fig.colorbar(im, cax=cbar_ax)
# cbar.set_label('Shear Modulus (GPa)')

# plt.suptitle(f'Vertical Slices Along Strike (constant X) - CG{CG_mu_deg}', fontsize=14)
# plt.tight_layout(rect=[0, 0, 0.88, 0.96])
# plt.show()

In [ ]:
# # Vertical slices along strike - μ anomaly
# fig, axes = plt.subplots(2, 2, figsize=(15, 10))
# axes = axes.flatten()

# anom_max = 100  # percent

# for i, (x_km, x_m) in enumerate(zip(x_strike_km, x_strike_m)):
#     if i >= len(axes):
#         break
        
#     ax = axes[i]
    
#     # Extract slice
#     y, z, mu_vals = extract_vertical_slice_strike(mu_3d_grid, x_m, 
#                                                    tolerance=slice_tolerance_s)
    
#     if len(y) < 10:
#         ax.set_title(f'X = {x_km} km\n(insufficient data)')
#         ax.axis('off')
#         continue
    
#     # Compute slice-specific reference mu (mean of this slice)
#     mu_ref_slice = mu_vals.mean()
    
#     # Compute anomaly relative to slice mean
#     mu_anomaly = (mu_vals - mu_ref_slice) / mu_ref_slice * 100
    
#     # Interpolate to regular grid
#     yi, zi_grid, anom_interp = interpolate_to_grid(y, z, mu_anomaly, grid_resolution=grid_res)
    
#     # Plot
#     im = ax.pcolormesh(yi/1e3, zi_grid/1e3, anom_interp, cmap='cmc.roma', 
#                        vmin=-anom_max, vmax=anom_max, shading='auto')
#     ax.set_xlabel('Y (km)')
#     ax.set_ylabel('Z (km)')
#     ax.set_title(f'X = {x_km} km\n(ref={mu_ref_slice:.1f} GPa, {len(y)} pts)')
#     ax.set_xlim([y_min, y_max])
#     ax.set_ylim([z_min_s, z_max_s])
#     ax.set_aspect('equal')

# # Colorbar
# fig.subplots_adjust(right=0.88)
# cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
# cbar = fig.colorbar(im, cax=cbar_ax)
# cbar.set_label('μ Anomaly (%)')

# plt.suptitle(f'Vertical Slices Along Strike - μ Anomaly (slice-specific ref, CG{CG_mu_deg})', fontsize=14)
# plt.tight_layout(rect=[0, 0, 0.88, 0.96])
# plt.show()

## 7. PyGMT Publication-Quality Plots

Higher quality plots using PyGMT for publication, with geographic coordinates, plate interface contours, and trench line - matching the style of the original notebook.

In [ ]:
# ============================================================================
# Helper function: Extract horizontal slice and convert to geographic coords
# Updated: Support for uneven mesh top boundary (z_top_interp parameter)
# ============================================================================

def extract_horizontal_slice_geo(grid, z_depth, tolerance=1500, field_name='shear modulus',
                                  z_top_interp=None):
    """
    Extract a horizontal slice and convert to geographic coordinates.

    Parameters:
    -----------
    grid : pv.PolyData or pv.UnstructuredGrid
        Grid or point cloud with field values
    z_depth : float
        Z coordinate of slice (meters, negative for depth)
    tolerance : float
        Half-thickness for point cloud filtering (meters)
    field_name : str
        Name of the field to extract
    z_top_interp : LinearNDInterpolator or None
        Interpolator for top boundary z(x,y). If provided, points above
        top boundary are masked (set to NaN). Use for uneven mesh.

    Returns:
    --------
    lon, lat : ndarray
        Geographic coordinates
    z_pts : ndarray
        Z coordinates of extracted points (metres, negative)
    values : ndarray
        Field values at those points (NaN for points above top boundary)
    """
    # Extract slice in mesh coordinates (pass z_top_interp for masking)
    x, y, z_pts, values = extract_horizontal_slice(grid, z_depth, tolerance, field_name,
                                            z_top_interp=z_top_interp)

    if len(x) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([])

    # Convert mesh coordinates to geographic
    # x, y are in mesh local coordinates; add offset then transform
    lon, lat = utp.ckm2LLd(x + x0, y + y0, lon0, lat0, -rot)

    return lon, lat, z_pts, values


# ============================================================================
# Multi-panel horizontal slice plot (matching matplotlib style)
# Updated: Support for uneven mesh top boundary (z_top_interp parameter)
# ============================================================================

def plot_horizontal_slices_pygmt(grid, depth_levels_km,
                                  tolerance=1500, plot_type='mu',
                                  nrows=2, ncols=2,
                                  cmap=None, series=None,
                                  lon_spacing=0.02, lat_spacing=0.02,
                                  margin_x=0.1, margin_y=0.25, 
                                  z_top_interp=None,
                                  ref_type='slice_mean',
                                  filename=None):
    """
    Create multi-panel PyGMT figure of horizontal slices.

    Matches the matplotlib section settings:
    - Slice-specific reference mu for anomaly
    - Color scale: 5-160 GPa for absolute, ±100% for anomaly
    - cmc.roma colormap

    Parameters:
    -----------
    grid : pv.PolyData or pv.UnstructuredGrid
        Grid or point cloud with shear modulus data
    depth_levels_km : list
        Depths to plot (km, positive values)
    tolerance : float
        Slice half-thickness (meters)
    plot_type : str
        'mu' for absolute values, 'anomaly' for percent anomaly
    nrows, ncols : int
        Subplot grid dimensions
    cmap : str, optional
        PyGMT colormap name (default: 'roma')
    series : list, optional
        Color scale [min, max] or [min, max, interval] for makecpt
        Default: [5, 160] for 'mu', [-100, 100] for 'anomaly'
    lon_spacing, lat_spacing : float
        Grid spacing in degrees for interpolation (default: 0.02)
    z_top_interp : LinearNDInterpolator or None
        Interpolator for top boundary z(x,y). If provided:
        - Points above top boundary are masked (left blank)
        For even mesh (z_top=0 everywhere), pass None to skip masking.
    ref_type : str, optional
        Reference for anomaly computation: 'slice_mean' uses the spatial
        mean of each slice; '1d_model' uses scalar 1D reference at the query depth
        (mu_1d_ref_method flag in cell 3 controls step vs. interp lookup).
    filename : str, optional
        Save figure to this path
    """
    # Figure size - each panel ~7cm x 8cm (matching reference notebook style)
    panel_width = 7  # cm
    panel_height = 8  # cm
    # margin_x = 0.1  # cm (horizontal spacing)
    # margin_y = 0.25  # cm (vertical spacing)
    fig_width_val = ncols * panel_width + (ncols - 1) * margin_x
    fig_height_val = nrows * panel_height + (nrows - 1) * margin_y
    fig_width = f"{fig_width_val}c"
    fig_height = f"{fig_height_val}c"

    # Colorbar dimensions
    cbar_width = panel_width * 0.8  # 80% of panel width
    cbar_height = 0.2  # cm

    # Colormap settings (use defaults if not specified)
    if cmap is None:
        cmap = 'roma'  # PyGMT name for cmc.roma

    if series is None:
        if plot_type == 'anomaly':
            series = [-100, 100]  # ±100% to match matplotlib
        else:
            series = [5, 160]  # 5-160 GPa to match matplotlib

    # Label for colorbar
    if plot_type == 'anomaly':
        if ref_type == '1d_model':
            label = 'μ anomaly rel. 1D (%)'
        else:
            label = 'μ anomaly (%)'
    else:
        label = 'μ (GPa)'

    # Panel labels
    panel_labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)', '(i)']

    fig = pygmt.Figure()

    # Create colormap once (will be used by all panels)
    pygmt.makecpt(cmap=cmap, series=series, background=True)

    with fig.subplot(nrows=nrows, ncols=ncols, figsize=(fig_width, fig_height),
                     autolabel=False, sharex=True, sharey=True,
                     margins=f"{margin_x}c/{margin_y}c", frame=["WSne"]):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                     MAP_TITLE_OFFSET="-0.15c", FONT_ANNOT_PRIMARY="8p")

        for idx_panel, depth_km in enumerate(depth_levels_km):
            if idx_panel >= nrows * ncols:
                break

            row = idx_panel // ncols
            col = idx_panel % ncols
            depth_m = -depth_km * 1e3  # Convert to meters (negative)

            with fig.set_panel(panel=[row, col]):
                fig.basemap(region=region_fault, projection="M?", frame=["a1f0.2"])

                # Extract slice in geographic coordinates (pass z_top_interp)
                lon_pts, lat_pts, z_pts, mu_vals = extract_horizontal_slice_geo(
                    grid, depth_m, tolerance=tolerance, z_top_interp=z_top_interp
                )

                if len(lon_pts) < 10:
                    fig.text(x=np.mean(region_fault[:2]), y=np.mean(region_fault[2:]),
                             text=f"No data at {depth_km} km", font="10p")
                    continue

                # Compute values to plot (handle NaN from masking)
                if plot_type == 'anomaly':
                    if ref_type == '1d_model':
                        # Scalar 1D reference at query depth (smoother display)
                        mu_ref_at_slice = float(mu_1d_ref_at_depths(np.array([depth_m]))[0])
                        values = (mu_vals - mu_ref_at_slice) / mu_ref_at_slice * 100
                        depth_label = f"Depth = {depth_km:.0f} km (1D ref={mu_ref_at_slice:.1f} GPa)"
                    else:
                        # Slice-specific reference mu (use nanmean to ignore masked values)
                        mu_ref_slice = np.nanmean(mu_vals)
                        depth_label = f"Depth = {depth_km:.0f} km (ref={mu_ref_slice:.0f} GPa)"
                        values = (mu_vals - mu_ref_slice) / mu_ref_slice * 100
                else:
                    values = mu_vals
                    depth_label = f"Depth = {depth_km:.0f} km"

                # Filter to region
                mask = (
                    (lon_pts >= region_fault[0]) & (lon_pts <= region_fault[1]) &
                    (lat_pts >= region_fault[2]) & (lat_pts <= region_fault[3])
                )
                lon_filt = lon_pts[mask]
                lat_filt = lat_pts[mask]
                z_filt   = z_pts[mask]
                val_filt = values[mask]

                # Remove NaN values for gridding (they will appear as blank areas)
                valid_vals = ~np.isnan(val_filt)
                lon_valid = lon_filt[valid_vals]
                lat_valid = lat_filt[valid_vals]
                val_valid = val_filt[valid_vals]

                print(val_valid.min(), val_valid.max())

                if len(lon_valid) < 10:
                    fig.text(x=np.mean(region_fault[:2]), y=np.mean(region_fault[2:]),
                             text=f"Insufficient data at {depth_km} km", font="10p")
                    continue

                # Block-median then surface for smooth gridding
                xyz = np.column_stack([lon_valid, lat_valid, val_valid])
                filtered = pygmt.blockmedian(data=xyz, region=region_fault, spacing=f"{lon_spacing}d")
                data_grid = pygmt.surface(data=filtered, region=region_fault,
                                          spacing=(lon_spacing, lat_spacing), tension=0.25)

                # Mask grid nodes above the top boundary (prevent surface extrapolation artifacts)
                if z_top_interp is not None:
                    lon_g = data_grid.x.values
                    lat_g = data_grid.y.values
                    lon_gg, lat_gg = np.meshgrid(lon_g, lat_g)
                    # Convert lon/lat grid nodes to mesh coordinates
                    x_rot, y_rot = utp.LL2ckmd(lon_gg.ravel(), lat_gg.ravel(), lon0, lat0, rot)
                    xy_q = np.column_stack([x_rot - x0, y_rot - y0])
                    z_top_at_grid = z_top_interp(xy_q).reshape(lon_gg.shape)
                    # depth_m is negative; mask where depth > z_top (above boundary)
                    above = (depth_m > z_top_at_grid) | np.isnan(z_top_at_grid)
                    data_grid.values[above] = np.nan

                # Plot the grid
                fig.grdimage(grid=data_grid, cmap=True, nan_transparent=True)

                # Coastline
                fig.coast(shorelines="0.5p,black", area_thresh=100)

                # Add plate interface contours (if plate_grd is defined)
                try:
                    fig.grdcontour(grid=plate_grd, levels=5, limit="-100/-10",
                                annotation="20+f6p", pen="0.4p,darkgray")
                    fig.grdcontour(grid=plate_grd, levels=[-depth_km],
                                annotation="20+f6p", pen="1.5p,darkgray")
                except:
                    pass  # Skip if plate_grd not available

                fig.plot(x=trench['lon'], y=trench['lat'], pen="1.5p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")

                # Plot vertical slice locations on first panel (if defined)
                try:
                    # Along-dip slices (constant y, red dashed)
                    for y_km in y_dip_km:
                        y_m = y_km * 1e3
                        x_line = np.linspace(x_min * 1e3, x_max * 1e3, 5)
                        y_line = np.full_like(x_line, y_m)
                        lon_line, lat_line = utp.ckm2LLd(x_line + x0, y_line + y0,
                                                        lon0, lat0, -rot)
                        fig.plot(x=lon_line, y=lat_line, pen="0.8p,white,--")
                        # Label at the downdip end (x_max side)
                        fig.text(x=lon_line[-1], y=lat_line[-1],
                                 text=f"{y_km:.0f} km",
                                 font="7p,Helvetica-Bold,white",
                                 justify="LM", offset="0.08c/0")
                except:
                    pass  # Skip if slice locations not defined

                # Depth label
                fig.text(x=region_fault[1] - 0.05, y=region_fault[3] - 0.05,
                         text=depth_label,
                         font="9p,Helvetica-Bold,white", justify="TR",
                         fill="gray60", offset="0.1c/0.1c")

                # Panel label
                fig.text(text=panel_labels[idx_panel],
                         x=region_fault[0] + 0.1, y=region_fault[3] - 0.08,
                         font="11p,Helvetica-Bold,black", justify="CM")

                # # Horizontal colorbar at bottom of first-row panels only
                # if row == 0:
                #     with pygmt.config(FONT_ANNOT_PRIMARY="9p", FONT_LABEL="10p"):
                #         if len(series) < 3:
                #             fig.colorbar(frame=["a50f10", f"x+l{label}"],
                #                          position=f"JBC+w{cbar_width:.1f}c/{cbar_height}c+h+o0/0.25c")
                #         else:
                #             fig.colorbar(frame=[f"a{series[2]*4}f{series[2]}", f"x+l{label}"],
                #                     position=f"JBC+w{cbar_width:.1f}c/{cbar_height}c+h+o0/0.25c")

    # share a single horizontal colorbar 
    cbar_width = fig_width_val * 0.7
    cbar_height = 0.25
    cbar_x = fig_width_val / 2   # centered horizontally
    cbar_y = fig_height_val / 2 + 0.5  # centered vertically (between rows)
    # Extension triangles show out-of-range colors (for anomaly plots only)
    ext_str = "+e0.2c" if plot_type == 'anomaly' else ""

    with pygmt.config(FONT_ANNOT_PRIMARY="8p", FONT_LABEL="10p"):
        if len(series) < 3:
            fig.colorbar(frame=["a10f5", f"x+l{label}"],
                         position=f"x{cbar_x}c/{cbar_y}c+w{cbar_width:.1f}c/{cbar_height}c+h+jTC{ext_str}")
        else:
            fig.colorbar(frame=[f"a{series[2]*4}f{series[2]}", f"x+l{label}"],
                         position=f"x{cbar_x}c/{cbar_y}c+w{cbar_width:.1f}c/{cbar_height}c+h+jTC{ext_str}")

    if filename:
        # fig.savefig(filename)
        fig.savefig(filename.replace('.pdf', '.png'), dpi=600)
        print(f"Saved: {filename.replace('.pdf', '.png')}")

    fig.show()
    return fig


print("PyGMT plotting functions defined (consistent with matplotlib style):")
print("  - Color scale: customizable via 'series' parameter (default: 5-160 GPa for absolute, ±100% for anomaly)")
print("  - Colormap: customizable via 'cmap' parameter (default: 'roma')")
print("  - Grid spacing: customizable via 'lon_spacing', 'lat_spacing' (default: 0.02°)")
print("  - z_top_interp parameter: masks points above top boundary (for uneven mesh)")
print("  - ref_type parameter: 'slice_mean' (default) or '1d_model' (scalar 1D ref at query depth, controlled by mu_1d_ref_method flag)")
print("  - Horizontal colorbar at bottom of first-row panels")

In [ ]:
# Plot horizontal slices - absolute μ values
depth_levels = [20, 30, 40, 50]  # km
# x_min, x_max = -100, 100   # km
# x_min, x_max = -80, 60   # km
x_min, x_max = -100, 80   # km

fig_mu = plot_horizontal_slices_pygmt(
    mu_3d_grid,
    depth_levels_km=depth_levels,
    tolerance=500,  # matching matplotlib section, chosen
    # tolerance=200,  # for testing
    plot_type='mu',
    lon_spacing=0.02, lat_spacing=0.02,
    cmap='roma',
    # series=[5, 160],    
    # series=[5, 160, (160-5)/50],
    series=[20, 80],
    # series=[5, 110],    # for testing
    nrows=2, ncols=2,
    margin_x=0.1, margin_y=0.4, 
    z_top_interp=z_top_interp,  # Pass for uneven mesh (None for even mesh)
    # filename=resultpath + f'mu_h_slices_{meshname}_CG{CG_mu_deg}.pdf' if flag_savefig else None
    filename=resultpath + f'mu_h_slices_{meshname}{het3d_str}.pdf' if flag_savefig else None
    
)

In [ ]:
# Plot horizontal slices - μ anomaly relative to 1D depth-dependent model
fig_anom_1d = plot_horizontal_slices_pygmt(
    mu_3d_grid,
    depth_levels_km=depth_levels,
    tolerance=500,  # chosen
    # tolerance=200,  # testing, sort of acceptable for scaling by 2.5 times 
    plot_type='anomaly',
    ref_type='1d_model',
    lon_spacing=0.02, lat_spacing=0.02,
    cmap='roma',
    series=[-40, 40],   # for original model
    # series=[-160, 160],   # for amplified model relative to 1D 
    # series=[-100, 100],   # for testing, for scaling by 2.5 times 
    nrows=2, ncols=2,
    margin_x=0.1, margin_y=0.4,
    z_top_interp=z_top_interp,
    # filename=resultpath + f'mu_h_slices_anom1d_{meshname}_CG{CG_mu_deg}.pdf' if flag_savefig else None
    filename=resultpath + f'mu_h_slices_anom1d_{meshname}{het3d_str}.pdf' if flag_savefig else None
)

### 7.2 Vertical Slices (PyGMT)

Vertical cross-sections along dip (constant y) with plate interface overlay.

In [ ]:
# ============================================================================
# Vertical slice plotting with plate interface using PyGMT
# Following step 10.6 in plt_slip_mu_relation_CK.ipynb
# Supports both single-column and multi-column layouts
# Updated: Support for uneven mesh top boundary (z_top_interp parameter)
# ============================================================================

def plot_vertical_slices_dip_pygmt(grid, y_dip_km,
                                    tolerance=2500, plot_type='mu',
                                    nrows=2, ncols=2,
                                    cmap=None, series=None,
                                    x_spacing=1.0, z_spacing=1.0,
                                    x_range=(-80, 60), z_range=(-60, 0),
                                    depth_levels=None,
                                    margin_x=0.15, margin_y=0.7,
                                    z_top_interp=None,
                                    ref_type='slice_mean',
                                    filename=None):
    """
    Create multi-panel PyGMT figure of vertical slices along dip.

    Parameters:
    -----------
    grid : pv.PolyData or pv.UnstructuredGrid
        Grid or point cloud with shear modulus data
    y_dip_km : list
        Y positions for slices (km from origin)
    tolerance : float
        Slice half-thickness (meters)
    plot_type : str
        'mu' for absolute values, 'anomaly' for percent anomaly
    nrows, ncols : int
        Subplot grid dimensions
        - ncols=1: Single column with shared vertical colorbar on right
        - ncols>1: Multi-column with horizontal colorbars on top row
    cmap : str, optional
        PyGMT colormap name (default: 'roma')
    series : list, optional
        Color scale [min, max] or [min, max, interval] for makecpt
        Default: [5, 160] for 'mu', [-100, 100] for 'anomaly'
    x_spacing, z_spacing : float
        Grid spacing in km for interpolation (default: 1.0)
    x_range, z_range : tuple
        (min, max) range in km for x and z axes (default: (-80, 60), (-60, 0))
        Note: z_range should be (z_min, z_max) where z_min < z_max (e.g., (-60, 0))
              This puts z=0 at top and more negative values (deeper) at bottom
    depth_levels : list, optional
        Depth levels (in km, negative) to draw as horizontal dashed lines
    margin_x : float
        Horizontal margin between panels in cm (default: 0.15)
    margin_y : float
        Vertical margin between panels in cm (default: 0.6)
    z_top_interp : LinearNDInterpolator or None
        Interpolator for top boundary z(x,y). If provided:
        - Points above top boundary are masked (left blank)
        - Top boundary line is drawn as dashed black line
        For even mesh (z_top=0 everywhere), pass None to skip masking.
    ref_type : str, optional
        Reference for anomaly: 'slice_mean' uses nanmean of each panel;
        '1d_model' uses per-DOF mu_1d_ref_at_depths() at each DOF's own depth
        (appropriate for vertical slices; mu_1d_ref_method flag controls step vs. interp).
    filename : str, optional
        Save figure to this path
    """
    # Colormap settings (use defaults if not specified)
    if cmap is None:
        cmap = 'roma'

    if series is None:
        if plot_type == 'anomaly':
            series = [-100, 100]  # +/-100%
        else:
            series = [5, 160]  # 5-160 GPa

    # Label for colorbar
    if plot_type == 'anomaly':
        if ref_type == '1d_model':
            label = 'μ anomaly rel. 1D (%)'
        else:
            label = 'μ anomaly (%)'
    else:
        label = 'μ (GPa)'

    # Region for slices [xmin, xmax, zmin, zmax] in km
    x_min, x_max = x_range
    z_min, z_max = z_range
    region_xz = [x_min, x_max, z_min, z_max]

    # Calculate figure size to maintain equal aspect ratio
    # For equal aspect: panel_width/panel_height = x_range/z_range
    x_span = x_max - x_min  # e.g., 180 km
    z_span = z_max - z_min  # e.g., 60 km
    aspect_ratio = x_span / z_span  # (e.g., 180/60 = 3.0)

    # Set panel height and calculate width to maintain equal aspect (1 km = 1 km on plot)
    panel_height = 4.0  # cm
    panel_width = panel_height * aspect_ratio  # cm (e.g., 4 * 3.0 = 12 cm)

    # For single column layout, add extra space on right for vertical colorbar
    if ncols == 1:
        cbar_space = 1.8  # cm for colorbar on right
        fig_width_val = panel_width + cbar_space
    else:
        cbar_space = 0
        fig_width_val = ncols * panel_width + (ncols - 1) * margin_x

    fig_height_val = nrows * panel_height + (nrows - 1) * margin_y
    fig_width = f"{fig_width_val}c"
    fig_height = f"{fig_height_val}c"

    # Prepare plate interface for slicing (convert to mesh coordinates)
    df_plate_mesh = df_plate.copy()
    x_rot, y_rot = utp.LL2ckmd(df_plate['lon'].values, df_plate['lat'].values,
                               lon0, lat0, rot)
    df_plate_mesh['x_mesh'] = (x_rot - x0) / 1e3  # to km
    df_plate_mesh['y_mesh'] = (y_rot - y0) / 1e3  # to km
    df_plate_mesh['z_mesh'] = df_plate['z'].values  # already in km

    plate_xy = np.column_stack([df_plate_mesh['x_mesh'], df_plate_mesh['y_mesh']])
    plate_z = df_plate_mesh['z_mesh'].values

    # Panel labels
    panel_labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)', '(g)', '(h)', '(i)']

    fig = pygmt.Figure()

    # Create colormap once (will be used by all panels)
    pygmt.makecpt(cmap=cmap, series=series, background=True)

    with fig.subplot(nrows=nrows, ncols=ncols, figsize=(fig_width, fig_height),
                     autolabel=False, sharex=True, sharey=True, frame=["WSne"],
                     margins=f"{margin_x}c/{margin_y}c"):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold",
                     MAP_TITLE_OFFSET="0.3c", FONT_ANNOT_PRIMARY="8p",
                     MAP_FRAME_PEN       = '0.5p,black',
                     )

        for idx_panel, y_km in enumerate(y_dip_km):
            if idx_panel >= nrows * ncols:
                break

            row = idx_panel // ncols
            col = idx_panel % ncols
            y_m = y_km * 1e3  # Convert to meters

            with fig.set_panel(panel=[row, col]):
                # Basemap with Cartesian projection (equal aspect ratio)
                with pygmt.config(FONT_LABEL="9p"):
                    if row == nrows - 1 and col == 0:
                        # Bottom row and 1st column: show x-axis label
                        fig.basemap(region=region_xz, projection="X?",
                                    frame=["xa20f10+lAlong-dip (km)", "ya20f10", "WSne"])
                    elif row == nrows - 1 and col > 0:
                        # Bottom row and 1st column: show x-axis label
                        fig.basemap(region=region_xz, projection="X?",
                                    frame=["xa20f10+lAlong-dip (km)", "ya20f10", "wSne"])
                    elif row < nrows - 1 and col > 0:
                        fig.basemap(region=region_xz, projection="X?",
                                    frame=["xa20f10", "ya20f10", "wsne"])
                    elif row < nrows - 1 and col == 0:
                        # Other rows: no x-axis label
                        fig.basemap(region=region_xz, projection="X?",
                                    frame=["xa20f10", "ya20f10", "Wsne"])

                # Extract slice (pass z_top_interp for masking if available)
                x, z, mu_vals = extract_vertical_slice_dip(grid, y_m, tolerance=tolerance,
                                                           z_top_interp=z_top_interp)

                if len(x) < 10:
                    fig.text(x=np.mean([x_min, x_max]), y=np.mean([z_min, z_max]),
                             text=f"No data at Y={y_km} km", font="10p")
                    continue

                # Convert to km
                x_km = x / 1e3
                z_km = z / 1e3

                # Compute values to plot (handle NaN from masking)
                if plot_type == 'anomaly':
                    if ref_type == '1d_model':
                        # Per-DOF 1D reference using each point's own depth
                        mu_ref_pts = mu_1d_ref_at_depths(z)  # z in metres (negative)
                        values = (mu_vals - mu_ref_pts) / mu_ref_pts * 100
                    else:
                        # Slice-mean reference
                        mu_ref_slice = np.nanmean(mu_vals)
                        values = (mu_vals - mu_ref_slice) / mu_ref_slice * 100
                else:
                    values = mu_vals
                slice_label = f"Along-strike = {y_km:.0f} km"

                # Filter to region (keep NaN values for proper masking)
                mask = (
                    (x_km >= x_min) & (x_km <= x_max) &
                    (z_km >= z_min) & (z_km <= z_max)
                )
                x_filt = x_km[mask]
                z_filt = z_km[mask]
                val_filt = values[mask]

                # Remove NaN values for gridding (they will appear as blank areas)
                valid_vals = ~np.isnan(val_filt)
                x_valid = x_filt[valid_vals]
                z_valid = z_filt[valid_vals]
                val_valid = val_filt[valid_vals]

                if len(x_valid) < 10:
                    continue

                print(f"  Panel {idx_panel} (y={y_km} km): {len(x_valid)} valid points in region" +
                      (f" ({len(x_filt) - len(x_valid)} masked)" if len(x_filt) > len(x_valid) else ""))

                # Block-median then surface for smooth gridding
                xyz = np.column_stack([x_valid, z_valid, val_valid])
                filtered = pygmt.blockmedian(data=xyz, region=region_xz,
                                             spacing=f"{x_spacing}k/{z_spacing}k")
                data_grid = pygmt.surface(data=filtered, region=region_xz,
                                          spacing=(x_spacing, z_spacing), tension=0.25)

                # Mask grid nodes above the top boundary (prevent surface extrapolation artifacts)
                if z_top_interp is not None:
                    xg = data_grid.x.values   # along-dip grid nodes (km)
                    zg = data_grid.y.values   # depth grid nodes (km)
                    xx_g, zz_g = np.meshgrid(xg, zg)
                    # Query z_top in mesh coordinates (meters)
                    xy_q = np.column_stack([xx_g.ravel() * 1e3, np.full(xx_g.size, y_m)])
                    z_top_grid = z_top_interp(xy_q).reshape(xx_g.shape) / 1e3  # to km
                    # Mask: above top boundary OR outside interpolation domain
                    above = (zz_g > z_top_grid) | np.isnan(z_top_grid)
                    data_grid.values[above] = np.nan

                # Plot the grid (colormap already created)
                fig.grdimage(grid=data_grid, cmap=True, nan_transparent=True)

                # Add depth level lines if provided
                if depth_levels is not None:
                    for depth in depth_levels:
                        fig.plot(x=[region_xz[0], region_xz[1]], y=[depth, depth],
                                 pen="0.5p,white,--")

                # Draw top boundary line if z_top_interp is provided (uneven mesh)
                if z_top_interp is not None:
                    x_top_km, z_top_km = get_top_boundary_line_dip(y_m, x_range, z_top_interp)
                    # Filter to valid (non-NaN) points within region
                    valid_top = ~np.isnan(z_top_km) & (z_top_km >= z_min) & (z_top_km <= z_max)
                    if valid_top.sum() > 2:
                        fig.plot(x=x_top_km[valid_top], y=z_top_km[valid_top],
                                 pen="1.2p,black")

                # Plate interface at this y (interpolate z at constant y)
                x_line = np.linspace(x_min, x_max, 200)
                plate_at_y = griddata(plate_xy, plate_z,
                                      np.column_stack([x_line, np.full_like(x_line, y_km)]),
                                      method='linear')
                # Plot plate interface line
                valid = ~np.isnan(plate_at_y)
                if valid.sum() > 2:
                    fig.plot(x=x_line[valid], y=plate_at_y[valid], pen="2p,darkgray")

                # Y-axis label (Depth)
                if ncols == 1:
                    if col == 0:
                        with pygmt.config(FONT_LABEL="9p"):
                            fig.basemap(frame=["y+lDepth (km)"])
                else:
                    # Multi-column: label on left column
                    if col == 0:
                        with pygmt.config(FONT_LABEL="9p"):
                            fig.basemap(frame=["y+lDepth (km)"])

                # Along-strike label at top-right corner
                fig.text(x=region_xz[1] - 2, y=region_xz[3] - 2,
                         text=slice_label,
                         font="10p,Helvetica-Bold,white", justify="TR",
                         fill="gray60", offset="0.1c/0.1c")

                # Panel label at bottom-left
                fig.text(text=panel_labels[idx_panel],
                         x=region_xz[0] + 1, y=region_xz[2] + 3,
                         font="11p,Helvetica-Bold,black", justify="LB")


    # Extension triangles show out-of-range colors (for anomaly plots only)
    ext_str = "+e0.2c" if plot_type == 'anomaly' else ""

    # Single column layout: draw shared vertical colorbar OUTSIDE subplot
    if ncols == 1:
        cbar_length = fig_height_val * 0.7  # 70% of figure height
        cbar_thickness = 0.3  # cm
        cbar_x = panel_width + 0.4  # cm from left edge of figure
        cbar_y = fig_height_val / 2  # center vertically

        with pygmt.config(FONT_ANNOT_PRIMARY="9p", FONT_LABEL="11p"):
            if len(series) < 3:
                fig.colorbar(frame=["a10f5", f"x+l{label}"],
                             position=f"x{cbar_x}c/{cbar_y}c+w{cbar_length:.1f}c/{cbar_thickness}c+v+jML+o2.0c/0{ext_str}")
            else:
                fig.colorbar(frame=[f"a{series[2]*4}f{series[2]}", f"x+l{label}"],
                             position=f"x{cbar_x}c/{cbar_y}c+w{cbar_length:.1f}c/{cbar_thickness}c+v+jML+o2.0c/0{ext_str}")

    else:
        cbar_width = fig_width_val * 0.7
        cbar_height = 0.25
        cbar_x = fig_width_val / 2   # centered horizontally
        cbar_y = fig_height_val / 2 + 0.6  # centered vertically (between rows)
        with pygmt.config(FONT_ANNOT_PRIMARY="7p", FONT_LABEL="9p"):
            if len(series) < 3:
                fig.colorbar(frame=["a10f5", f"x+l{label}"],
                             position=f"x{cbar_x}c/{cbar_y}c+w{cbar_width:.1f}c/{cbar_height}c+h+jTC{ext_str}")
            else:
                fig.colorbar(frame=[f"a{series[2]*4}f{series[2]}", f"x+l{label}"],
                             position=f"x{cbar_x}c/{cbar_y}c+w{cbar_width:.1f}c/{cbar_height}c+h+jTC{ext_str}")


    if filename:
        # fig.savefig(filename)
        fig.savefig(filename.replace('.pdf', '.png'), dpi=600)
        print(f"Saved: {filename.replace('.pdf', '.png')}")

    fig.show()
    return fig


print("PyGMT vertical slice plotting function defined (flexible layout):")
print("  - Equal aspect ratio: panel width auto-calculated from x/z range ratio")
print("  - Z-axis: z=0 at top, more negative (deeper) at bottom")
print("  - Labels: 'Along-dip (km)' for x-axis, 'Depth (km)' for y-axis")
print("  - Adjustable margins: margin_x, margin_y parameters (in cm)")
print("  - z_top_interp parameter: masks points above top boundary & draws dashed line")
print("  - Layout options:")
print("    * ncols=1: Single column with shared vertical colorbar on right")
print("    * ncols>1: Multi-column with horizontal colorbars on top row")

In [ ]:
# Plot vertical slices along dip - absolute μ
y_slices = [-10, 10, 30, 50]  # km (matching matplotlib section 5)
depth_levels = [-20, -30, -40, -50]   # km

# Single column (4 rows × 1 col) - better for wide aspect ratios
fig = plot_vertical_slices_dip_pygmt(
    mu_3d_grid,
    y_dip_km=y_slices,
    # tolerance=2500,  # for testing
    # tolerance=1500,  # for testing
    # tolerance=1000,  # for testing
    tolerance=500,  # chosen
    plot_type='mu',
    x_spacing=1.0, z_spacing=1.0,
    # x_range=(-80, 60),
    # x_range=(-100, 100),
    x_range=(-100, 80),
    z_range=(-60, 0),
    # series=[5, 160],
    # series=[5, 160, (160-5)/50],
    series=[20, 80],    # for orginal, unscaled model
    # series=[5, 110],    # for testing
    nrows=4, ncols=1,  # Single column
    depth_levels=depth_levels,
    margin_x=0.15, margin_y=0.1,
    z_top_interp=z_top_interp,  # Pass for uneven mesh (None for even mesh)
    # filename=resultpath + f'mu_vdip_slices_1col_{meshname}_CG{CG_mu_deg}.pdf' if flag_savefig else None
    filename=resultpath + f'mu_vdip_slices_1col_{meshname}{het3d_str}.pdf' if flag_savefig else None
)

In [ ]:
# Plot vertical slices along dip - μ anomaly relative to 1D depth-dependent model
fig_v_anom1d = plot_vertical_slices_dip_pygmt(
    mu_3d_grid,
    y_dip_km=y_slices,
    tolerance=500,
    # tolerance=1000,
    plot_type='anomaly',
    ref_type='1d_model',
    x_spacing=1.0, z_spacing=1.0,
    x_range=(-100, 80),
    z_range=(-60, 0),
    # series=[-160, 160],
    series=[-40, 40],   # for original model
    # series=[-100, 100],   # for testing, for scaling by 2.5 times 
    nrows=4, ncols=1,  # Single column
    depth_levels=depth_levels,
    margin_x=0.15, margin_y=0.1,
    z_top_interp=z_top_interp,
    # filename=resultpath + f'mu_vdip_anom1d_1col_{meshname}_CG{CG_mu_deg}.pdf' if flag_savefig else None
    filename=resultpath + f'mu_vdip_anom1d_1col_{meshname}{het3d_str}.pdf' if flag_savefig else None
)


In [ ]:
# Multi-column (2 rows × 2 cols)
fig_v_mu = plot_vertical_slices_dip_pygmt(
    mu_3d_grid,
    y_dip_km=y_slices,
    # tolerance=2500,  # matching matplotlib section
    tolerance=1000,  # matching matplotlib section
    plot_type='mu',
    x_spacing=1.0, z_spacing=1.0,
    # x_range=(-80, 60),
    # x_range=(-100, 100),
    x_range=(-100, 80),
    z_range=(-60, 0),
    series=[20, 80],
    nrows=2, ncols=2,
    depth_levels=depth_levels,
    z_top_interp=z_top_interp,  # Pass for uneven mesh (None for even mesh)
    filename=resultpath + f'mu_vdip_slices_{meshname}_CG{CG_mu_deg}.pdf' if flag_savefig else None
)

# # Plot vertical slices - μ anomaly
# fig_v_anom = plot_vertical_slices_dip_pygmt(
#     mu_3d_grid,
#     y_dip_km=y_slices,
#     tolerance=1000,
#     plot_type='anomaly',
#     x_spacing=1.0, z_spacing=1.0,
#     x_range=(-100, 80),
#     z_range=(-60, 0),
#     nrows=2, ncols=2,
#     depth_levels=depth_levels,
#     z_top_interp=z_top_interp,  # Pass for uneven mesh (None for even mesh)
#     filename=resultpath + f'mu_vertical_dip_anomaly_CG{CG_mu_deg}.pdf'
# )

In [ ]:
# ── 3-Panel combined figure: (A) mesh cross-section | (B) μ dip | (C) μ horizontal ──
# Config — change these to switch slice position / depth / plot type
y_slice_km   = 30.0        # along-strike position for left panels (km)
mesh_geo_source = 'manual'   # 'manual' = tet-plane intersection; 'pyvista' = vtkCutter
depth_km_h   = 20.0        # depth for horizontal slice (positive km)
x_range_mesh = (-300, 300) # Panel A: full mesh along-dip range (km)
x_range_dip  = (-100, 80)  # Panel B: zoomed along-dip range (km)
z_mesh_range = (-200, 5)   # Panel A depth range (km)
z_mu_range   = (-60, 0)    # Panel B depth range (km)
tolerance_mu = 500         # μ extraction tolerance (m)
# plot_type_3p = 'mu'        # 'mu' or 'anomaly'
# series_3p    = [20, 80]    # color range: [min, max] GPa for 'mu'; e.g. [-40,40] for 'anomaly'
plot_type_3p = 'anomaly'        # 'mu' or 'anomaly'
series_3p    = [-40, 40]    # color range: [min, max] GPa for 'mu'; e.g. [-40,40] for 'anomaly'
cmap_3p      = 'roma'

# ── Layout math (equal-aspect left panels; height-matched right panel) ────
W_L      = 10.0  # left panel width (cm)
gap_AB   = 0.7   # gap between Panels A and B (cm)
# gap_LR   = 1.1   # gap between left and right columns (cm)
gap_LR   = 0.5   # gap between left and right columns (cm)
cbar_h   = 1.8    # colorbar zone below Panel C (bar ~0.25 + tick labels ~0.35 + label ~0.4 + gap)
cbar_gap = 0.85  # Panel C bottom → colorbar top (cm)
cbar_thick = 0.15

x_span_mesh = x_range_mesh[1] - x_range_mesh[0]  # 600 km
x_span_dip  = x_range_dip[1]  - x_range_dip[0]   # 180 km
H_A = W_L * (z_mesh_range[1] - z_mesh_range[0]) / x_span_mesh  # equal-aspect for Panel A
H_B = W_L * (z_mu_range[1]   - z_mu_range[0])   / x_span_dip   # equal-aspect for Panel B
H_LR = H_A + gap_AB + H_B   # total left-column height = Panel C height

# Mercator aspect: WGS84 ellipsoidal isometric latitude (matches GMT default)
_lat1, _lat2 = region[2], region[3]
_e = 0.0818191908426  # WGS84 eccentricity
def _iso_lat(lat_deg):
    lat_r = np.radians(lat_deg); s = np.sin(lat_r)
    return (np.log(np.tan(np.pi/4 + lat_r/2))
            - _e/2 * np.log((1 + _e*s) / (1 - _e*s)))
_merc_lat = _iso_lat(_lat2) - _iso_lat(_lat1)
W_R = H_LR / (_merc_lat / np.radians(region[1] - region[0]))

print(f"Panel A (mesh):   {W_L:.2f} × {H_A:.2f} cm")
print(f"Panel B (μ dip):  {W_L:.2f} × {H_B:.2f} cm")
print(f"Panel C (μ horiz):{W_R:.2f} × {H_LR:.2f} cm  (+ {cbar_h:.1f} cm colorbar below)")
print(f"Figure total:     {W_L+gap_LR+W_R:.2f} × {cbar_h+H_LR:.2f} cm")

# ── Mesh cross-section (Panel A) ──────────────────────────────────────────
import meshio as _meshio

_msh = _meshio.read(meshpath + meshname + '.msh')
_pts = np.asarray(_msh.points[:, :3], float)
_tets, _ttags, _tris, _tritags = [], [], [], []
for _i, _blk in enumerate(_msh.cells):
    _t = _msh.cell_data["gmsh:physical"][_i]
    if _blk.type == 'tetra':   _tets.append(_blk.data);  _ttags.append(_t)
    elif _blk.type == 'triangle': _tris.append(_blk.data); _tritags.append(_t)
_tets     = np.vstack(_tets).astype(np.int64);   _ttags   = np.concatenate(_ttags).astype(int)
_tris     = np.vstack(_tris).astype(np.int64);   _tritags = np.concatenate(_tritags).astype(int)

_TAG_TOP=1; _TAG_FAULT=7; _TAG_LEFT=8; _TAG_RIGHT=9
_EPAIRS = ((0,1),(0,2),(0,3),(1,2),(1,3),(2,3))
_y0_m   = y_slice_km * 1e3

def _dedup3(pts, tol=1.0):
    keep = []
    for p in pts:
        if all(np.linalg.norm(p-k) > tol for k in keep): keep.append(p)
    return keep

def _tet_poly(xyz, y0, tol=1.0):
    dy = xyz[:,1] - y0
    if np.all(dy>tol) or np.all(dy<-tol): return None
    pts = []
    for i,j in _EPAIRS:
        d0,d1 = dy[i],dy[j]
        if abs(d0)<=tol and abs(d1)<=tol: pts.extend([xyz[i],xyz[j]]); continue
        if abs(d0)<=tol: pts.append(xyz[i]); continue
        if abs(d1)<=tol: pts.append(xyz[j]); continue
        if d0*d1<0: t=d0/(d0-d1); pts.append(xyz[i]+t*(xyz[j]-xyz[i]))
    pts = _dedup3(pts)
    if len(pts)<3: return None
    xz = np.array(pts)[:,[0,2]]
    ctr = xz.mean(0); ang = np.arctan2(xz[:,1]-ctr[1], xz[:,0]-ctr[0])
    return xz[np.argsort(ang)] / 1e3

def _tri_seg(xyz, y0, tol=1.0):
    dy = xyz[:,1] - y0
    if np.all(dy>tol) or np.all(dy<-tol): return None
    pts = []
    for i,j in ((0,1),(0,2),(1,2)):
        d0,d1 = dy[i],dy[j]
        if abs(d0)<=tol and abs(d1)<=tol: pts.extend([xyz[i],xyz[j]]); continue
        if abs(d0)<=tol: pts.append(xyz[i]); continue
        if abs(d1)<=tol: pts.append(xyz[j]); continue
        if d0*d1<0: t=d0/(d0-d1); pts.append(xyz[i]+t*(xyz[j]-xyz[i]))
    pts = _dedup3(pts)
    if len(pts)<2: return None
    return np.array(pts[:2])[:,[0,2]] / 1e3

_yv = _pts[_tets,1]
_strad = (_yv.min(1)<=_y0_m) & (_yv.max(1)>=_y0_m)
_polys_L, _polys_R = [], []
for _idx, _tag in zip(_tets[_strad], _ttags[_strad]):
    _p = _tet_poly(_pts[_idx], _y0_m)
    if _p is None: continue
    if _tag==_TAG_LEFT:  _polys_L.append(_p)
    elif _tag==_TAG_RIGHT: _polys_R.append(_p)

_ftris = _tris[_tritags==_TAG_FAULT];  _fy = _pts[_ftris,1]
_fault_segs = []
for _idx in _ftris[(_fy.min(1)<=_y0_m) & (_fy.max(1)>=_y0_m)]:
    s = _tri_seg(_pts[_idx], _y0_m)
    if s is not None: _fault_segs.append(s)

_ttris = _tris[_tritags==_TAG_TOP];   _ty = _pts[_ttris,1]
_top_segs_mesh = []
for _idx in _ttris[(_ty.min(1)<=_y0_m) & (_ty.max(1)>=_y0_m)]:
    s = _tri_seg(_pts[_idx], _y0_m)
    if s is not None: _top_segs_mesh.append(s)

print(f"Mesh polys: {len(_polys_L)} left, {len(_polys_R)} right | "
      f"{len(_fault_segs)} fault segs, {len(_top_segs_mesh)} top segs")

if mesh_geo_source == 'pyvista':
    import pyvista as _pv
    # Build PyVista UnstructuredGrid from already-loaded mesh data
    _n_tets   = len(_tets)
    _cells_pv = np.hstack([np.full((_n_tets,1), 4, dtype=np.int64), _tets]).ravel()
    _types_pv = np.full(_n_tets, _pv.CellType.TETRA, dtype=np.uint8)
    _grid_pv  = _pv.UnstructuredGrid(_cells_pv, _types_pv, _pts)
    _grid_pv.cell_data['block'] = _ttags.astype(float)

    def _make_surf_pv(tri_idx):
        n = len(tri_idx)
        c = np.hstack([np.full((n,1),3,dtype=np.int64), tri_idx]).ravel()
        t = np.full(n, _pv.CellType.TRIANGLE, dtype=np.uint8)
        return _pv.UnstructuredGrid(c, t, _pts)

    _fault_surf_pv = _make_surf_pv(_tris[_tritags==_TAG_FAULT])
    _top_surf_pv   = _make_surf_pv(_tris[_tritags==_TAG_TOP])

    _pv_origin = [0.0, _y0_m, 0.0];  _pv_normal = [0.0, 1.0, 0.0]
    _sl_blocks = _grid_pv.cell_data_to_point_data().slice(normal=_pv_normal, origin=_pv_origin)
    _sl_fault  = _fault_surf_pv.slice(normal=_pv_normal, origin=_pv_origin)
    _sl_top    =   _top_surf_pv.slice(normal=_pv_normal, origin=_pv_origin)

    # Parse mixed tri/quad faces → polygon lists (km)
    _pts_km_pv  = _sl_blocks.points[:, [0, 2]] / 1e3
    _bvals      = _sl_blocks['block']
    _ff         = np.asarray(_sl_blocks.faces)
    _polys_L, _polys_R = [], []
    _ii = 0
    while _ii < len(_ff):
        _nn   = int(_ff[_ii])
        _vids = _ff[_ii+1 : _ii+1+_nn]
        _poly = _pts_km_pv[_vids]
        if _bvals[_vids].mean() < 8.5:  _polys_L.append(_poly)
        else:                           _polys_R.append(_poly)
        _ii += _nn + 1

    # Extract line segments from PyVista PolyData .lines
    def _pv_to_segs(poly):
        if poly.n_points == 0 or len(poly.lines) == 0: return []
        _p = poly.points[:, [0, 2]] / 1e3
        _fl = np.asarray(poly.lines); _out = []; _i = 0
        while _i < len(_fl):
            _nv = int(_fl[_i]); _vi = _fl[_i+1:_i+1+_nv]
            for _k in range(len(_vi)-1): _out.append(_p[[_vi[_k],_vi[_k+1]]])
            _i += _nv + 1
        return _out

    _fault_segs    = _pv_to_segs(_sl_fault)
    _top_segs_mesh = _pv_to_segs(_sl_top)
    print(f"PyVista polys: {len(_polys_L)} left, {len(_polys_R)} right | "
          f"{len(_fault_segs)} fault segs, {len(_top_segs_mesh)} top segs")

# ── GMT polygon/line helpers ──────────────────────────────────────────────
def _write_gmt_polys(polys, path, color_hex, alpha=0):
    fill = f'{color_hex}@{alpha}' if alpha>0 else color_hex
    with open(path,'w') as f:
        for poly in polys:
            f.write(f'> -G{fill}\n')
            for x,z in np.vstack([poly, poly[0]]):
                f.write(f'{x:.6f} {z:.6f}\n')

def _write_gmt_lines(segs, path):
    with open(path,'w') as f:
        for seg in segs:
            f.write('>\n')
            for x,z in seg: f.write(f'{x:.6f} {z:.6f}\n')

def _poly_iso_q(poly):
    # Isoperimetric quotient 4*pi*A/P^2: ~0.6-0.78 for well-shaped tris/quads, ->0 for slivers
    p = np.asarray(poly)
    x, z = p[:,0], p[:,1]
    A = 0.5*abs(np.dot(x, np.roll(z,-1)) - np.dot(z, np.roll(x,-1)))
    P = np.sum(np.linalg.norm(np.diff(np.vstack([p, p[0]]), axis=0), axis=1))
    return 4*np.pi*A/P**2 if P > 0 else 0.0

def _write_gmt_poly_edges(polys, path):
    # Closed-polygon outlines only (no fill flag), for stroking element edges
    with open(path,'w') as f:
        for poly in polys:
            f.write('>\n')
            for x,z in np.vstack([poly, poly[0]]):
                f.write(f'{x:.6f} {z:.6f}\n')

# ── μ data for Panel B (dip slice) ───────────────────────────────────────
y_m_slice = y_slice_km * 1e3
x_dip, z_dip, mu_dip = extract_vertical_slice_dip(
    mu_3d_grid, y_m_slice, tolerance=tolerance_mu, z_top_interp=z_top_interp)

if len(x_dip) > 0:
    x_km_dip, z_km_dip = x_dip/1e3, z_dip/1e3
    if plot_type_3p == 'anomaly':
        val_dip = (mu_dip - mu_1d_ref_at_depths(z_dip)) / mu_1d_ref_at_depths(z_dip) * 100
    else:
        val_dip = mu_dip
    x_min_d, x_max_d = x_range_dip;  z_min_d, z_max_d = z_mu_range
    mask_d = ((x_km_dip>=x_min_d)&(x_km_dip<=x_max_d)&
              (z_km_dip>=z_min_d)&(z_km_dip<=z_max_d)&~np.isnan(val_dip))
    region_dip = [x_min_d, x_max_d, z_min_d, z_max_d]
    xyz_dip = np.column_stack([x_km_dip[mask_d], z_km_dip[mask_d], val_dip[mask_d]])
    _filt_dip  = pygmt.blockmedian(data=xyz_dip, region=region_dip, spacing="1k/1k")
    grid_dip   = pygmt.surface(data=_filt_dip, region=region_dip, spacing=(1,1), tension=0.25)
    if z_top_interp is not None:
        xg,zg = grid_dip.x.values, grid_dip.y.values
        xx_g,zz_g = np.meshgrid(xg,zg)
        xy_q = np.column_stack([xx_g.ravel()*1e3, np.full(xx_g.size,y_m_slice)])
        z_top_g = z_top_interp(xy_q).reshape(xx_g.shape)/1e3
        grid_dip.values[(zz_g>z_top_g)|np.isnan(z_top_g)] = np.nan
    print(f"Panel B: {mask_d.sum()} DOFs gridded for μ dip slice at y={y_slice_km} km")

# ── μ data for Panel C (horizontal slice) ────────────────────────────────
depth_m_h = -depth_km_h * 1e3
lon_h, lat_h, z_h, mu_h = extract_horizontal_slice_geo(
    mu_3d_grid, depth_m_h, tolerance=tolerance_mu, z_top_interp=z_top_interp)

if len(lon_h) > 0:
    if plot_type_3p == 'anomaly':
        mu_ref_h = float(mu_1d_ref_at_depths(np.array([depth_m_h]))[0])
        val_h = (mu_h - mu_ref_h) / mu_ref_h * 100
    else:
        val_h = mu_h
    mask_h = ((lon_h>=region[0])&(lon_h<=region[1])&
              (lat_h>=region[2])&(lat_h<=region[3])&~np.isnan(val_h))
    xyz_h = np.column_stack([lon_h[mask_h], lat_h[mask_h], val_h[mask_h]])
    _filt_h  = pygmt.blockmedian(data=xyz_h, region=region, spacing="0.02d")
    grid_h   = pygmt.surface(data=_filt_h, region=region, spacing=(0.02,0.02), tension=0.25)
    if z_top_interp is not None:
        lon_g,lat_g = grid_h.x.values, grid_h.y.values
        lon_gg,lat_gg = np.meshgrid(lon_g,lat_g)
        x_rot,y_rot = utp.LL2ckmd(lon_gg.ravel(),lat_gg.ravel(),lon0,lat0,rot)
        xy_q = np.column_stack([x_rot-x0, y_rot-y0])
        z_top_g = z_top_interp(xy_q).reshape(lon_gg.shape)
        grid_h.values[(depth_m_h>z_top_g)|np.isnan(z_top_g)] = np.nan
    print(f"Panel C: {mask_h.sum()} DOFs gridded for μ horiz slice at {depth_km_h} km depth")

# Plate interface in mesh coords (needed for Panel B)
_df_pm = df_plate.copy()
_xr,_yr = utp.LL2ckmd(df_plate['lon'].values, df_plate['lat'].values, lon0, lat0, rot)
plate_xy_km = np.column_stack([(_xr-x0)/1e3, (_yr-y0)/1e3])
plate_z_km  = df_plate['z'].values

# Import GNSS data, originally from Feng et al. 2012, but no volcano sites, both trench-parallel and normal components
obs_disp_name = "/CKfig6_data_final.csv"   # the EXACT data file for figure 6 in Kyriakopoulos et al. (2016)

# the processed data has the unit of m/yr that was converted from mm/yr
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', \
                          'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])


In [ ]:
# ── Assemble 3-panel figure ───────────────────────────────────────────────
fig3p = pygmt.Figure()

pygmt.makecpt(cmap=cmap_3p, series=series_3p, background=True)

pygmt.config(MAP_FRAME_TYPE='plain', FONT_ANNOT_PRIMARY='7p',
             FONT_LABEL='8p', MAP_FRAME_PEN='0.5p,black')

_tmp3 = {k: f'/tmp/gmt_3p_{k}.txt' for k in ('left','right','fault','topm')}

# ── Origin: start at bottom of Panel B (leave cbar_h below for colorbar) ──
fig3p.shift_origin(yshift=f'{cbar_h}c')

# ═══ Panel B: μ dip slice (bottom-left) ══════════════════════════════════
region_B  = [x_range_dip[0], x_range_dip[1], z_mu_range[0], z_mu_range[1]]
proj_B    = f'X{W_L}c/{H_B:.3f}c'
fig3p.basemap(region=region_B, projection=proj_B,
              frame=['WSne', 'xa20f10+lAlong-dip (km)', 'ya20f10+lDepth (km)'])

if len(x_dip) > 0:
    fig3p.grdimage(grid=grid_dip, cmap=True, nan_transparent=True)
    # Plate interface
    x_line = np.linspace(x_range_dip[0], x_range_dip[1], 200)
    p_at_y = griddata(plate_xy_km, plate_z_km,
                      np.column_stack([x_line, np.full_like(x_line, y_slice_km)]),
                      method='linear')
    valid = ~np.isnan(p_at_y)
    if valid.sum() > 2:
        fig3p.plot(x=x_line[valid], y=p_at_y[valid], pen='2p,darkgray')
    # Depth lines
    for _d in [-20]:
        if z_mu_range[0] <= _d <= z_mu_range[1]:
            fig3p.plot(x=list(x_range_dip), y=[_d,_d], pen='1p,white,--')
    # Top boundary
    if z_top_interp is not None:
        _xtk, _ztk = get_top_boundary_line_dip(y_m_slice, x_range_dip, z_top_interp)
        _vt = ~np.isnan(_ztk) & (_ztk>=z_mu_range[0]) & (_ztk<=z_mu_range[1])
        if _vt.sum() > 2:
            fig3p.plot(x=_xtk[_vt], y=_ztk[_vt], pen='1.2p,black')

# GPS stations at z=0: convert lon/lat to along-dip coords
_gps_x_rot, _gps_y_rot = utp.LL2ckmd(df['lon'].values, df['lat'].values, lon0, lat0, rot)
_gps_x_km = (_gps_x_rot - x0) / 1e3   # along-dip (km)
in_range = ((_gps_x_km >= x_range_dip[0]) & (_gps_x_km <= x_range_dip[1]))
if in_range.sum() > 0:
    fig3p.plot(x=_gps_x_km[in_range], y=np.zeros(in_range.sum())+2.0,
               style='i0.2c', fill='white', pen='0.5p,black', no_clip=True)

fig3p.text(x=x_range_dip[0]+1, y=z_mu_range[0]+2,
           text='(b)', font='10p,Helvetica-Bold,black', justify='LB')



# ═══ Shift up to Panel A ═════════════════════════════════════════════════
fig3p.shift_origin(yshift=f'{H_B + gap_AB}c')

# ═══ Panel A: mesh cross-section (top-left) ═══════════════════════════════
region_A = [x_range_mesh[0], x_range_mesh[1], z_mesh_range[0], z_mesh_range[1]]
proj_A   = f'X{W_L}c/{H_A:.3f}c'
fig3p.basemap(region=region_A, projection=proj_A,
              frame=['WSne', 'xa100f50', 'ya50f25+lDepth (km)'])  #+lAlong-dip (km)

_elem_pen = '0.12p,gray35'   # mesh element edges: soft gray softens the 'distortion' look vs harsh black
_sliver_q = 0.0             # drop edges of cut polygons below this isoperimetric quotient (slivers); fill kept
_tmp3['leftE']  = '/tmp/gmt_3p_leftE.txt'
_tmp3['rightE'] = '/tmp/gmt_3p_rightE.txt'
if _polys_L:
    _write_gmt_polys(_polys_L, _tmp3['left'],  '#4878CF', alpha=45)
    fig3p.plot(data=_tmp3['left'])                       # fill pass: all polys, no stroke (continuous region)
    _write_gmt_poly_edges([p for p in _polys_L if _poly_iso_q(p) >= _sliver_q], _tmp3['leftE'])
    fig3p.plot(data=_tmp3['leftE'], pen=_elem_pen)       # edge pass: well-shaped polys only
if _polys_R:
    _write_gmt_polys(_polys_R, _tmp3['right'], '#D65F5F', alpha=45)
    fig3p.plot(data=_tmp3['right'])
    _write_gmt_poly_edges([p for p in _polys_R if _poly_iso_q(p) >= _sliver_q], _tmp3['rightE'])
    fig3p.plot(data=_tmp3['rightE'], pen=_elem_pen)
if _top_segs_mesh:
    _write_gmt_lines(_top_segs_mesh, _tmp3['topm'])
    fig3p.plot(data=_tmp3['topm'], pen='1.2p,black')
if _fault_segs:
    _write_gmt_lines(_fault_segs, _tmp3['fault'])
    fig3p.plot(data=_tmp3['fault'], pen='2p,darkgray')

# Legend for mesh panels
_leg_3p = '/tmp/gmt_3p_legend.txt'
with open(_leg_3p,'w') as _f:
    _f.write(
            'S 0.4c r 0.3c #4878CF@45 0.5p,white 0.8c Cocos Plate\n'
            'S 0.4c r 0.3c #D65F5F@45 0.5p,white 0.8c Caribbean Plate\n'
            'S 0.4c - 0.5c - 2p,darkgray 0.8c Plate interface\n'
            'S 0.4c - 0.5c - 1.2p,black 0.8c Top boundary\n'
    )
with pygmt.config(FONT_ANNOT_PRIMARY="6p"):
    fig3p.legend(spec=_leg_3p, position='JBR+jBR+o0.1c', box='+gwhite+p0.4p,gray60')

fig3p.text(x=x_range_mesh[1]-5, y=z_mesh_range[1]-8,
           text=f'Along-strike = {y_slice_km:g} km', font='7p,Helvetica-Bold,white', justify='TR',
           fill="gray60", offset="0.1c/0.1c")
fig3p.text(x=x_range_mesh[0]+5, y=z_mesh_range[0]+10,
           text='(a)', font='10p,Helvetica-Bold,black', justify='LB')

# Rectangle showing Panel B extent
_rx = [x_range_dip[0], x_range_dip[1], x_range_dip[1], x_range_dip[0], x_range_dip[0]]
_rz = [z_mu_range[0],  z_mu_range[0],  z_mu_range[1],  z_mu_range[1],  z_mu_range[0]]
fig3p.plot(x=_rx, y=_rz, pen='0.8p,chartreuse,-.')


# ═══ Shift to Panel C (right panel, bottom-aligned with Panel B) ══════════
fig3p.shift_origin(xshift=f'{W_L + gap_LR}c', yshift=f'-{H_B + gap_AB}c')

# ═══ Panel C: horizontal μ slice (right) ═════════════════════════════════
proj_C = f'M{W_R:.3f}c'
fig3p.basemap(region=region, projection=proj_C, frame=['ESnw', 'a1f0.2'])

if len(lon_h) > 0:
    fig3p.grdimage(grid=grid_h, cmap=True, nan_transparent=True)
    fig3p.coast(shorelines='0.5p,black', area_thresh=100)
    try:
        fig3p.grdcontour(grid=plate_grd, levels=5, limit='-100/-10',
                         annotation='20+f6p', pen='0.4p,darkgray')
        fig3p.grdcontour(grid=plate_grd, levels=[-depth_km_h], pen='1.5p,darkgray')
    except: pass
    fig3p.plot(x=trench['lon'], y=trench['lat'], pen='1.5p,dimgray',
               style='f0.4i/0.075i+l+t', fill='dimgray')
    # Along-dip slice location
    _xl = np.linspace(x_range_dip[0]*1e3, x_range_dip[1]*1e3, 50)
    _lon_l, _lat_l = utp.ckm2LLd(_xl+x0, np.full_like(_xl,y_m_slice)+y0, lon0, lat0, -rot)
    fig3p.plot(x=_lon_l, y=_lat_l, pen='1p,white,--')

# Depth label
_depth_lbl = f'Depth = {depth_km_h:.0f} km'
if plot_type_3p == 'anomaly':
    _depth_lbl += f' (1D ref={float(mu_1d_ref_at_depths(np.array([depth_m_h]))[0]):.1f} GPa)'
fig3p.text(x=region[1]-0.05, y=region[3]-0.05, text=_depth_lbl,
           font='7p,Helvetica-Bold,white', justify='TR', fill='gray60', offset='0.1c/0.1c')
fig3p.text(x=region[0]+0.1, y=region[3]-0.08, text='(c)',
           font='10p,Helvetica-Bold,black', justify='CM')

# ── Colorbar: horizontal, centered below Panel C ──────────────────────────
_cbar_len = W_R * 0.5
_lbl = 'μ (GPa)' if plot_type_3p == 'mu' else 'μ anomaly rel. 1D (%)'
_ext = '+e0.2c' if plot_type_3p == 'anomaly' else ''
with pygmt.config(FONT_ANNOT_PRIMARY='12p', FONT_LABEL='13p'):
    fig3p.colorbar(
        frame=['a10f5', f'x+l{_lbl}'],
        position=f'JBC+jTC+o-1.2c/-{cbar_gap:.2f}c+w{_cbar_len:.1f}c/{cbar_thick}c+h{_ext}'
    )

fig3p.show()


In [ ]:
# # Save 3-panel figure
# _fname_3p = (resultpath +
#     f'mu_3panel_{meshname}{het3d_str}_{plot_type_3p}'
#     f'_y{y_slice_km:.0f}km_dep{depth_km_h:.0f}km.png')
# if flag_savefig:
#     fig3p.savefig(_fname_3p, dpi=600)
#     print(f'Saved: {_fname_3p}')


In [ ]:
# ── Quad-triangulation variant of Panel A (separate figure for comparison) ──
# A plane cutting a tetrahedron yields a triangle OR a quadrilateral; neither is a
# real 2D element, so splitting cut-quads into triangles is no less honest and gives
# a uniform triangular appearance. (Does NOT change skewed-triangle shapes.)
def _triangulate_poly(poly):
    p = np.asarray(poly)
    n = len(p)
    if n <= 3:
        return [p]
    return [np.array([p[0], p[i], p[i+1]]) for i in range(1, n-1)]   # fan from vertex 0

_polys_L_tri = [t for poly in _polys_L for t in _triangulate_poly(poly)]
_polys_R_tri = [t for poly in _polys_R for t in _triangulate_poly(poly)]
print(f'Triangulated: {len(_polys_L)}->{len(_polys_L_tri)} left, {len(_polys_R)}->{len(_polys_R_tri)} right polygons')

# ── Assemble 3-panel figure (quad-triangulated Panel A) ──
fig3p_tri = pygmt.Figure()

pygmt.makecpt(cmap=cmap_3p, series=series_3p, background=True)

pygmt.config(MAP_FRAME_TYPE='plain', FONT_ANNOT_PRIMARY='7p',
             FONT_LABEL='8p', MAP_FRAME_PEN='0.5p,black')

_tmp3 = {k: f'/tmp/gmt_3p_{k}.txt' for k in ('left','right','fault','topm')}

# ── Origin: start at bottom of Panel B (leave cbar_h below for colorbar) ──
fig3p_tri.shift_origin(yshift=f'{cbar_h}c')

# ═══ Panel B: μ dip slice (bottom-left) ══════════════════════════════════
region_B  = [x_range_dip[0], x_range_dip[1], z_mu_range[0], z_mu_range[1]]
proj_B    = f'X{W_L}c/{H_B:.3f}c'
fig3p_tri.basemap(region=region_B, projection=proj_B,
              frame=['WSne', 'xa20f10+lAlong-dip (km)', 'ya20f10+lDepth (km)'])

if len(x_dip) > 0:
    fig3p_tri.grdimage(grid=grid_dip, cmap=True, nan_transparent=True)
    # Plate interface
    x_line = np.linspace(x_range_dip[0], x_range_dip[1], 200)
    p_at_y = griddata(plate_xy_km, plate_z_km,
                      np.column_stack([x_line, np.full_like(x_line, y_slice_km)]),
                      method='linear')
    valid = ~np.isnan(p_at_y)
    if valid.sum() > 2:
        fig3p_tri.plot(x=x_line[valid], y=p_at_y[valid], pen='2p,darkgray')
    # Depth lines
    for _d in [-20]:
        if z_mu_range[0] <= _d <= z_mu_range[1]:
            fig3p_tri.plot(x=list(x_range_dip), y=[_d,_d], pen='1p,white,--')
    # Top boundary
    if z_top_interp is not None:
        _xtk, _ztk = get_top_boundary_line_dip(y_m_slice, x_range_dip, z_top_interp)
        _vt = ~np.isnan(_ztk) & (_ztk>=z_mu_range[0]) & (_ztk<=z_mu_range[1])
        if _vt.sum() > 2:
            fig3p_tri.plot(x=_xtk[_vt], y=_ztk[_vt], pen='1.2p,black')

# GPS stations at z=0: convert lon/lat to along-dip coords
_gps_x_rot, _gps_y_rot = utp.LL2ckmd(df['lon'].values, df['lat'].values, lon0, lat0, rot)
_gps_x_km = (_gps_x_rot - x0) / 1e3   # along-dip (km)
in_range = ((_gps_x_km >= x_range_dip[0]) & (_gps_x_km <= x_range_dip[1]))
if in_range.sum() > 0:
    fig3p_tri.plot(x=_gps_x_km[in_range], y=np.zeros(in_range.sum())+2.0,
               style='i0.2c', fill='white', pen='0.5p,black', no_clip=True)

fig3p_tri.text(x=x_range_dip[0]+1, y=z_mu_range[0]+2,
           text='(b)', font='10p,Helvetica-Bold,black', justify='LB')



# ═══ Shift up to Panel A ═════════════════════════════════════════════════
fig3p_tri.shift_origin(yshift=f'{H_B + gap_AB}c')

# ═══ Panel A: mesh cross-section (top-left) ═══════════════════════════════
region_A = [x_range_mesh[0], x_range_mesh[1], z_mesh_range[0], z_mesh_range[1]]
proj_A   = f'X{W_L}c/{H_A:.3f}c'
fig3p_tri.basemap(region=region_A, projection=proj_A,
              frame=['WSne', 'xa100f50', 'ya50f25+lDepth (km)'])  #+lAlong-dip (km)

_elem_pen = '0.12p,gray35'   # mesh element edges
_sliver_q_tri = 0.25          # 0 = show all triangle edges; raise to drop sliver triangles from split quads
_tmp3['left']   = '/tmp/gmt_3p_tri_left.txt'
_tmp3['right']  = '/tmp/gmt_3p_tri_right.txt'
_tmp3['leftE']  = '/tmp/gmt_3p_tri_leftE.txt'
_tmp3['rightE'] = '/tmp/gmt_3p_tri_rightE.txt'
if _polys_L_tri:
    _write_gmt_polys(_polys_L_tri, _tmp3['left'],  '#4878CF', alpha=45)
    fig3p_tri.plot(data=_tmp3['left'])                       # fill pass: all triangles, no stroke
    _write_gmt_poly_edges([p for p in _polys_L_tri if _poly_iso_q(p) >= _sliver_q_tri], _tmp3['leftE'])
    fig3p_tri.plot(data=_tmp3['leftE'], pen=_elem_pen)       # edge pass: triangle edges
if _polys_R_tri:
    _write_gmt_polys(_polys_R_tri, _tmp3['right'], '#D65F5F', alpha=45)
    fig3p_tri.plot(data=_tmp3['right'])
    _write_gmt_poly_edges([p for p in _polys_R_tri if _poly_iso_q(p) >= _sliver_q_tri], _tmp3['rightE'])
    fig3p_tri.plot(data=_tmp3['rightE'], pen=_elem_pen)
if _top_segs_mesh:
    _write_gmt_lines(_top_segs_mesh, _tmp3['topm'])
    fig3p_tri.plot(data=_tmp3['topm'], pen='1.2p,black')
if _fault_segs:
    _write_gmt_lines(_fault_segs, _tmp3['fault'])
    fig3p_tri.plot(data=_tmp3['fault'], pen='2p,darkgray')

# Legend for mesh panels
_leg_3p = '/tmp/gmt_3p_legend.txt'
with open(_leg_3p,'w') as _f:
    _f.write(
            'S 0.4c r 0.3c #4878CF@45 0.5p,white 0.8c Cocos Plate\n'
            'S 0.4c r 0.3c #D65F5F@45 0.5p,white 0.8c Caribbean Plate\n'
            'S 0.4c - 0.5c - 2p,darkgray 0.8c Plate interface\n'
            'S 0.4c - 0.5c - 1.2p,black 0.8c Top boundary\n'
    )
with pygmt.config(FONT_ANNOT_PRIMARY="6p"):
    fig3p_tri.legend(spec=_leg_3p, position='JBR+jBR+o0.1c', box='+gwhite+p0.4p,gray60')

fig3p_tri.text(x=x_range_mesh[1]-5, y=z_mesh_range[1]-8,
           text=f'Along-strike = {y_slice_km:g} km', font='7p,Helvetica-Bold,white', justify='TR',
           fill="gray60", offset="0.1c/0.1c")
fig3p_tri.text(x=x_range_mesh[0]+5, y=z_mesh_range[0]+10,
           text='(a)', font='10p,Helvetica-Bold,black', justify='LB')

# Rectangle showing Panel B extent
_rx = [x_range_dip[0], x_range_dip[1], x_range_dip[1], x_range_dip[0], x_range_dip[0]]
_rz = [z_mu_range[0],  z_mu_range[0],  z_mu_range[1],  z_mu_range[1],  z_mu_range[0]]
fig3p_tri.plot(x=_rx, y=_rz, pen='0.8p,chartreuse,-.')


# ═══ Shift to Panel C (right panel, bottom-aligned with Panel B) ══════════
fig3p_tri.shift_origin(xshift=f'{W_L + gap_LR}c', yshift=f'-{H_B + gap_AB}c')

# ═══ Panel C: horizontal μ slice (right) ═════════════════════════════════
proj_C = f'M{W_R:.3f}c'
fig3p_tri.basemap(region=region, projection=proj_C, frame=['ESnw', 'a1f0.2'])

if len(lon_h) > 0:
    fig3p_tri.grdimage(grid=grid_h, cmap=True, nan_transparent=True)
    fig3p_tri.coast(shorelines='0.5p,black', area_thresh=100)
    try:
        fig3p_tri.grdcontour(grid=plate_grd, levels=5, limit='-100/-10',
                         annotation='20+f6p', pen='0.4p,darkgray')
        fig3p_tri.grdcontour(grid=plate_grd, levels=[-depth_km_h], pen='1.5p,darkgray')
    except: pass
    fig3p_tri.plot(x=trench['lon'], y=trench['lat'], pen='1.5p,dimgray',
               style='f0.4i/0.075i+l+t', fill='dimgray')
    # Along-dip slice location
    _xl = np.linspace(x_range_dip[0]*1e3, x_range_dip[1]*1e3, 50)
    _lon_l, _lat_l = utp.ckm2LLd(_xl+x0, np.full_like(_xl,y_m_slice)+y0, lon0, lat0, -rot)
    fig3p_tri.plot(x=_lon_l, y=_lat_l, pen='1p,white,--')

# Depth label
_depth_lbl = f'Depth = {depth_km_h:.0f} km'
if plot_type_3p == 'anomaly':
    _depth_lbl += f' (1D ref={float(mu_1d_ref_at_depths(np.array([depth_m_h]))[0]):.1f} GPa)'
fig3p_tri.text(x=region[1]-0.05, y=region[3]-0.05, text=_depth_lbl,
           font='7p,Helvetica-Bold,white', justify='TR', fill='gray60', offset='0.1c/0.1c')
fig3p_tri.text(x=region[0]+0.1, y=region[3]-0.08, text='(c)',
           font='10p,Helvetica-Bold,black', justify='CM')

# ── Colorbar: horizontal, centered below Panel C ──────────────────────────
_cbar_len = W_R * 0.5
_lbl = 'μ (GPa)' if plot_type_3p == 'mu' else 'μ anomaly rel. 1D (%)'
_ext = '+e0.2c' if plot_type_3p == 'anomaly' else ''
with pygmt.config(FONT_ANNOT_PRIMARY='12p', FONT_LABEL='13p'):
    fig3p_tri.colorbar(
        frame=['a10f5', f'x+l{_lbl}'],
        position=f'JBC+jTC+o-1.2c/-{cbar_gap:.2f}c+w{_cbar_len:.1f}c/{cbar_thick}c+h{_ext}'
    )

fig3p_tri.show()


In [ ]:
# Save 3-panel figure
_fname_3p_tri = (resultpath +
    f'mu_3panel_tri_{meshname}{het3d_str}_{plot_type_3p}'
    f'_y{y_slice_km:.0f}km_dep{depth_km_h:.0f}km.png')
if flag_savefig:
    fig3p_tri.savefig(_fname_3p_tri, dpi=600)
    print(f'Saved: {_fname_3p_tri}')

## Summary

This notebook provides a unified approach to visualize shear modulus data for both CG1 and CG2 elements, with publication-quality plotting matching the original notebook style.

### File Format

**CG1 files:**
- `mu_true_..._DeShon3D_ref_4.xdmf` + `.h5` - Standard FEniCS XDMF export

**CG2 files (new format):**
- `mu_true_..._CGmu2.xdmf` + `.h5` - Standard XDMF (vertex values only, for ParaView)
- `mu_true_..._CGmu2_dofs.h5` - **NEW**: Contains ALL DOF coordinates and values (vertices + edge midpoints)

### Loading
- **CG1**: Loaded from XDMF as `UnstructuredGrid` with cell topology
- **CG2**: Loaded from `_dofs.h5` as `PolyData` point cloud with ALL DOF locations
  - Falls back to XDMF if `_dofs.h5` not found (but only gets vertex values)

### Slicing
- **CG1**: Uses PyVista's `.slice()` for exact plane-mesh intersection
- **CG2**: Uses tolerance-based filtering (necessary since point clouds have no cell topology)

### Interpolation
- Both use `griddata` to interpolate scattered slice points to regular grids (matching original)

### PyGMT Publication Plots
- Geographic coordinates (lon/lat) with proper coordinate transformation
- Plate interface depth contours overlaid
- Trench line with teeth symbols
- Panel labels for multi-panel figures
- Colormaps matching original: `gist_rainbow_r` for μ, `cmc.roma` for anomaly

### Key Functions
- `load_xdmf_as_pyvista(filepath, cg_degree)`: Load XDMF/H5 with CG1 or CG2 support
- `extract_horizontal_slice(grid, z_depth, tolerance)`: Unified horizontal slice extraction
- `extract_horizontal_slice_geo(grid, z_depth, tolerance)`: With geographic coordinate conversion
- `extract_vertical_slice_dip/strike(grid, coord, tolerance)`: Vertical slice extraction
- `plot_horizontal_slices_pygmt(grid, depths, ...)`: Multi-panel PyGMT horizontal slices
- `plot_vertical_slices_dip_pygmt(grid, y_slices, ...)`: Multi-panel vertical slices with plate interface

### Key Advantage of CG2
- ~6x more sample points for the same mesh (vertices + edge midpoints)
- Denser sampling leads to smoother interpolation results
- Better representation of continuous fields without the "grainy" artifacts of CG1

### Workflow
1. Run `test_higher_CGmu_synth_stripeslip_3DCK_noi.py` with `CG_mu_deg = 2`
2. This generates both `.xdmf` and `_dofs.h5` files
3. This notebook loads from `_dofs.h5` to get the full CG2 point density